# Notebook 7: Cross-Target Meta-Analysis and Results Assembly

**Purpose:** integrate what notebooks 2, 3, 3B, 4, 5 and 6 already computed into
the cross-target answers this project exists to give. **No new modelling, no new
docking and no new screening happens here** -- every number reported below is
read from disk, aligned across targets, and rendered into paper/SI-ready tables
and figures.

The four questions this notebook answers:

1. Does the same workflow perform consistently across GPCRs? (Module B)
2. Which target properties predict model performance? (Module C)
3. Which features transfer across GPCRs? (Module D)
4. Does uncertainty behave consistently across targets? (Module E)

Plus two integrative sections -- zero-shot transfer (Module F) and the combined
screening/docking evidence funnel (Module G) -- then figures, tables and a
SHA-256 manifest (Module H).

**Input severity is deliberately split into two levels.** Outputs from notebooks
2, 3, 3B and 6 are REQUIRED: if they are missing the notebook stops, because
Modules B-F cannot be honestly assembled without them. Outputs from notebooks 4
and 5 (DrugBank screening, docking) are OPTIONAL and feed Module G only; if they
are absent the notebook warns, skips Module G, and still produces everything
else. A late-finishing docking run must not block the rest of the results
assembly.

**Placeholder-input guard.** An earlier prototype of this analysis was assembled
against a scaffolding version of the external-validation summary in which every
target carried identical dummy values (ROC-AUC 0.9, n 15/300, Brier 0.1). It
produced a plausible-looking cross-target table that was entirely fictitious.
Module A therefore tests required inputs for degenerate constant columns and
refuses to run rather than repeating that silently.

**Deployment provenance is inherited, not re-selected here.** For classification,
targets represented in Notebook 4's corrected primary DrugBank artifact
`drugbank_primary_ranked_candidates.csv` inherit the exact FULL-pool + COMBINED
classifier stamped in that artifact. Targets absent from that primary screen
(currently CCR5) fall back to Notebook 3's leak-free
`best_algorithm_by_combination.csv`. Regression continues to use Notebook 3's
leak-free selection table. No algorithm is selected in this notebook from
held-out test performance, and ambiguous upstream provenance is a hard failure.


In [ ]:
# MUST BE FIRST CELL!
import os
import multiprocessing

HPC_MODE = False

def _running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

COLAB_MODE = (not HPC_MODE) and _running_in_colab()

if HPC_MODE:
    N_CORES = int(os.environ.get("NCPUS") or os.environ.get("PBS_NP") or
                  os.environ.get("PBS_NCPUS") or os.environ.get("SLURM_CPUS_PER_TASK") or
                  multiprocessing.cpu_count())
    import matplotlib
    matplotlib.use("Agg")
else:
    N_CORES = min(multiprocessing.cpu_count(), 4)

for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[var] = "1" if HPC_MODE else str(N_CORES)
os.environ["OMP_NESTED"] = "FALSE"
os.environ["MKL_DYNAMIC"] = "FALSE"

ENV = "HPC" if HPC_MODE else ("Colab" if COLAB_MODE else "local")
print(f"Environment: {ENV} | workers: {N_CORES}")

In [ ]:
from pathlib import Path

if HPC_MODE:
    PROJECT_DIR = Path("./").resolve()
elif COLAB_MODE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/My Drive/gpcr_benchmark")
else:
    PROJECT_DIR = Path.cwd().resolve()

data_path = PROJECT_DIR / "data"
processed_path = data_path / "processed"
results_path = PROJECT_DIR / "ml" / "results"
meta_path = results_path / "meta_analysis"
screen_path = results_path / "drugbank_screening"
ext_path = results_path / "external_validation"
zs_path = results_path / "zeroshot_transfer"
dockres_dir = PROJECT_DIR / "docking" / "docking_results"

out_fig_dir = PROJECT_DIR / "outputs" / "figures"
out_tab_dir = PROJECT_DIR / "outputs" / "tables"
for d in (meta_path, out_fig_dir, out_tab_dir):
    d.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"Meta-analysis outputs: {meta_path}")
print(f"Figures: {out_fig_dir}")
print(f"Tables:  {out_tab_dir}")

In [ ]:
import json
import hashlib
import datetime
import platform
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats

# =============================================================================
# PUBLICATION FIGURE STYLE -- identical convention to notebooks 02/03/04: no
# on-figure titles (the caption carries that), no code identifiers in any label
# or legend, American spelling, 300dpi PNG + vector PDF, panel letters via
# panel_label() rather than descriptive titles.
# =============================================================================
mpl.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Colorblind-safe palette, same colors used from notebook 02 onward.
BLUE, GREY, RED, ORANGE = "#1f77b4", "#c9c9c9", "#d62728", "#ff7f0e"
GREEN, PURPLE = "#2ca02c", "#9467bd"
PALETTE = [BLUE, ORANGE, GREEN, RED, PURPLE]

all_outputs = {}

def _register(name, path_obj, n_rows=None):
    entry = {"path": str(path_obj)}
    if n_rows is not None:
        entry["n_rows"] = int(n_rows)
    all_outputs[name] = entry

def save_table(df, stem, out_dir=None):
    """Every statistic this notebook reports is written to disk at the moment it
    is computed, never left only in cell output."""
    out_dir = Path(out_dir) if out_dir is not None else meta_path
    out_dir.mkdir(parents=True, exist_ok=True)
    p = out_dir / f"{stem}.csv"
    df.to_csv(p, index=False)
    _register(f"{stem}.csv", p, len(df))
    print(f"  saved: {p.name}  ({len(df)} rows)")
    return p

def save_fig(fig, stem, source_df=None):
    """Figures are never saved image-only. The dataframe a figure was drawn from
    is written beside it as <stem>_source.csv, so any panel can be regenerated or
    re-styled later without re-deriving the numbers."""
    for ext in ("png", "pdf"):
        p = out_fig_dir / f"{stem}.{ext}"
        fig.savefig(p)
        _register(f"{stem}.{ext}", p)
    plt.close(fig)
    if source_df is not None:
        sp = out_fig_dir / f"{stem}_source.csv"
        source_df.to_csv(sp, index=False)
        _register(f"{stem}_source.csv", sp, len(source_df))
        print(f"  saved: {stem}.png/.pdf + {sp.name}")
    else:
        print(f"  saved: {stem}.png/.pdf")

def panel_label(ax, letter):
    """Bold panel letter (A/B/C...), top-left -- NOT a descriptive title."""
    ax.set_title(letter, loc="left", fontweight="bold", fontsize=12)

def _sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("Imports and figure style OK.")
print("Python:", platform.python_version())
for pkg in ("numpy", "pandas", "matplotlib", "scipy"):
    mod = __import__(pkg)
    print(f"{pkg}: {getattr(mod, '__version__', 'unknown')}")

## Module A: Configuration and Input Audit

Loads every upstream artifact this notebook consumes, records what was found in
`meta_analysis_input_audit.csv`, and enforces the two-level severity rule:
required inputs hard-fail, optional inputs only disable Module G.

Three integrity checks run here because each one, if skipped, would corrupt every
downstream table silently rather than loudly:

1. **Placeholder detection** -- a required per-target metric column that is
   constant across all five targets is treated as scaffolding, not data.
2. **Target coverage** -- all five targets must appear in every required
   per-target table, so no two tables compare different panels.
3. **Deployed-algorithm agreement** -- the algorithm an upstream notebook
   recorded as deployed is compared against the leak-free selection table, and
   any disagreement is reported rather than resolved silently.

In [ ]:
# =============================================================================
# CONFIGURATION -- must match notebooks 2/3 exactly.
# =============================================================================
TARGETS = ["drd2", "cb2", "adora2a", "oprm1", "ccr5"]
TARGET_LABELS = {"drd2": "DRD2", "cb2": "CB2", "adora2a": "ADORA2A",
                 "oprm1": "OPRM1", "ccr5": "CCR5"}

# The deployed configuration the paper reports on: full activity pool, combined
# feature representation. Other pools and representations are still analysed
# (Module D compares representations directly), but every headline cross-target
# claim is made at this one configuration so all five targets are compared on
# identical footing.
POOL = "full"
REPRESENTATION = "combined"
CONFIDENCE_LEVEL = 0.9

# An external-validation row with fewer than this many compounds is reported but
# never presented as an equal-strength claim beside a large-n row.
MIN_N_FOR_PRIMARY_CLAIM = 30

print(f"Targets: {TARGETS}")
print(f"Headline configuration: pool={POOL}, representation={REPRESENTATION}, "
      f"conformal confidence={CONFIDENCE_LEVEL}")

In [ ]:
# =============================================================================
# INPUT REGISTRY -- (key, path, severity, module fed).
#   required -> hard fail if missing/empty/unreadable
#   optional -> warn only; the Module G group additionally disables Module G
# =============================================================================
INPUTS = [
    # ---- notebook 2 (data characterization) ----
    ("dataset_summary",         processed_path / "dataset_summary_all_targets_pools.csv", "required", "C"),
    ("scaffold_diversity",      processed_path / "scaffold_diversity_summary.csv",        "required", "C"),
    ("assay_support",           processed_path / "assay_support_summary.csv",             "required", "C"),
    ("endpoint_composition",    processed_path / "endpoint_composition_summary.csv",      "required", "C"),
    ("measurement_support",     processed_path / "measurement_support_distribution.csv",  "optional", "C"),
    ("structural_alerts",       processed_path / "structural_alert_summary.csv",          "optional", "C"),
    # ---- notebook 3 (ML benchmark) ----
    ("best_algorithm",          results_path / "best_algorithm_by_combination.csv",       "required", "B"),
    ("final_performance",       results_path / "final_model_test_performance.csv",        "required", "B/D"),
    ("calibration",             results_path / "calibration_summary.csv",                 "required", "B/E"),
    ("conformal",               results_path / "conformal_prediction_summary.csv",        "required", "B/E"),
    ("test_predictions",        results_path / "deployed_model_test_predictions.csv",     "required", "B/E"),
    ("shap_summaries",          results_path / "shap_summaries.json",                     "required", "D"),
    ("scaffold_breakdown",      results_path / "scaffold_performance_breakdown.csv",      "optional", "C/E"),
    ("difficulty_analysis",     results_path / "difficulty_analysis.csv",                 "optional", "C"),
    ("bootstrap_ci",            results_path / "bootstrap_confidence_intervals.csv",      "optional", "B"),
    ("statistical_comparisons", results_path / "statistical_comparisons.csv",             "optional", "D"),
    ("learning_curves",         results_path / "learning_curves.csv",                     "optional", "C"),
    ("persistent_errors",       results_path / "persistent_errors.csv",                   "optional", "E"),
    ("reliability_curves",      results_path / "reliability_curves.csv",                  "optional", "E"),
    # ---- notebook 3B (zero-shot transfer) ----
    ("zeroshot_headline",       zs_path / "zeroshot_transfer_headline.csv",               "required", "F"),
    ("zeroshot_metrics",        zs_path / "zeroshot_transfer_metrics.csv",                "optional", "F"),
    # ---- notebook 6 (external validation) ----
    ("external_validation",     ext_path / "external_validation_summary.csv",             "required", "B/E"),
    # ---- notebook 4 (DrugBank screening) -- Module G only ----
    ("screening_novel_summary", screen_path / "analysis_summary_novel_by_target.csv",     "optional", "G"),
    ("screening_candidates",    screen_path / "drugbank_primary_ranked_candidates.csv",   "optional", "G"),
    ("screening_polypharm",     screen_path / "cross_target_polypharmacology_hits.csv",   "optional", "G"),
    # ---- notebook 5 (docking) -- Module G only ----
    ("docking_redocking",       dockres_dir / "redocking_validation.csv",                 "optional", "G"),
    ("docking_candidate_sel",   dockres_dir / "docking_candidate_selection_summary.csv",  "optional", "G"),
    ("docking_final",           dockres_dir / "docking_results_final.csv",                "optional", "G"),
    ("docking_discriminative",  dockres_dir / "discriminative_validation.csv",            "optional", "G"),
    ("docking_ml_correlation",  dockres_dir / "ml_docking_correlation.csv",               "optional", "G"),
    ("docking_rediscovery",     dockres_dir / "docking_side_rediscovery_check.csv",       "optional", "G"),
    ("docking_property_bias",   dockres_dir / "enrichment_property_bias.csv",             "optional", "G"),
    ("docking_enrichment_set",  dockres_dir / "enrichment_validation_set.csv",            "optional", "G"),
]

DATA = {}
audit_records = []
missing_required, missing_optional = [], []

for key, path, severity, module in INPUTS:
    exists = Path(path).exists()
    n_rows, note, obj = None, "", None
    if exists:
        try:
            if str(path).endswith(".json"):
                with open(path) as fh:
                    obj = json.load(fh)
                n_rows = len(obj)
            else:
                obj = pd.read_csv(path)
                n_rows = len(obj)
            if not n_rows:
                note, obj = "file present but empty", None
        except Exception as exc:
            note, obj = f"unreadable: {type(exc).__name__}: {exc}", None
    else:
        note = "not found"

    if obj is not None:
        DATA[key] = obj
    else:
        (missing_required if severity == "required" else missing_optional).append((key, str(path), note))

    audit_records.append({
        "input_key": key, "path": str(path), "severity": severity,
        "feeds_module": module, "found": exists, "n_rows": n_rows, "note": note,
    })

input_audit_df = pd.DataFrame(audit_records)
save_table(input_audit_df, "meta_analysis_input_audit")

print()
for key, path, note in missing_optional:
    print(f"  warning: optional input unavailable -- {key} ({note})")
if missing_required:
    print()
    for key, path, note in missing_required:
        print(f"  ERROR: required input unavailable -- {key}: {path} ({note})")
    raise FileNotFoundError(
        f"{len(missing_required)} required input(s) unavailable. Modules B-F cannot be "
        f"assembled without them. Run the upstream notebook(s), or re-sync their results "
        f"from wherever they were produced, then rerun this notebook.")

n_required = sum(1 for r in audit_records if r["severity"] == "required")
print(f"\nAll {n_required} required inputs present.")

In [ ]:
# =============================================================================
# INTEGRITY CHECK 1 -- placeholder/scaffolding detection.
#
# A required per-target metric column that takes exactly one distinct value
# across all five targets is almost certainly scaffolding rather than a real
# run: five independently trained models on five different datasets do not
# produce byte-identical metrics. This check exists because a prototype of this
# analysis was once assembled from an external-validation file in which every
# target carried ROC-AUC 0.9, n = 15/300 and Brier 0.1, and the resulting
# cross-target table looked entirely reasonable.
# =============================================================================
PLACEHOLDER_CHECKS = [
    ("external_validation", ["roc_auc", "n", "brier"], "notebook 6"),
    ("final_performance",   ["test_roc_auc"],          "notebook 3"),
    ("calibration",         ["brier_calibrated"],      "notebook 3"),
]

placeholder_records = []
for key, cols, produced_by in PLACEHOLDER_CHECKS:
    df = DATA.get(key)
    if df is None:
        continue
    for col in cols:
        if col not in df.columns:
            continue
        vals = pd.to_numeric(df[col], errors="coerce").dropna()
        if len(vals) < 2:
            continue
        suspicious = bool(vals.nunique() == 1)
        placeholder_records.append({
            "input_key": key, "produced_by": produced_by, "column": col,
            "n_values": len(vals), "n_unique": int(vals.nunique()),
            "constant_value": float(vals.iloc[0]) if suspicious else np.nan,
            "looks_like_placeholder": suspicious,
        })

placeholder_df = pd.DataFrame(placeholder_records)
save_table(placeholder_df, "meta_analysis_placeholder_check")

flagged = placeholder_df[placeholder_df["looks_like_placeholder"]] if len(placeholder_df) else pd.DataFrame()
if len(flagged):
    print()
    print(flagged[["input_key", "produced_by", "column", "constant_value", "n_values"]].to_string(index=False))
    raise ValueError(
        "Placeholder-looking input detected: the column(s) above are constant across every "
        "row. That is what scaffolding data looks like, not what five independently trained "
        "models produce. Re-sync the real outputs from wherever the run actually executed "
        "before assembling any cross-target result.")
print("Placeholder check passed: no required metric column is constant across targets.")

In [ ]:
# =============================================================================
# INTEGRITY CHECK 2 -- target coverage in every required per-target table.
# =============================================================================
coverage_records = []
for key in ["final_performance", "calibration", "conformal", "test_predictions",
            "external_validation", "dataset_summary", "scaffold_diversity"]:
    df = DATA.get(key)
    if df is None or "target" not in getattr(df, "columns", []):
        continue
    present = sorted(set(df["target"].astype(str)) & set(TARGETS))
    missing = sorted(set(TARGETS) - set(present))
    coverage_records.append({
        "input_key": key, "n_targets_present": len(present),
        "targets_present": ",".join(present), "targets_missing": ",".join(missing),
        "complete": len(missing) == 0,
    })

coverage_df = pd.DataFrame(coverage_records)
save_table(coverage_df, "meta_analysis_target_coverage")

incomplete = coverage_df[~coverage_df["complete"]] if len(coverage_df) else pd.DataFrame()
if len(incomplete):
    print()
    print(incomplete.to_string(index=False))
    raise ValueError(
        "A required per-target table is missing one or more targets. A cross-target "
        "comparison assembled from an incomplete panel would silently compare different "
        "target sets in different tables.")
print("Target-coverage check passed: all 5 targets present in every required table.")

In [ ]:
# =============================================================================
# DEPLOYED ALGORITHM PROVENANCE -- inherited from the actual production path.
#
# Classification provenance hierarchy:
#   1) Notebook 4 corrected primary DrugBank artifact, when that target is
#      represented there. That artifact must be FULL + COMBINED and contain
#      exactly one algorithm per target.
#   2) Notebook 3 best_algorithm_by_combination.csv only for targets absent
#      from Notebook 4's primary screen (currently CCR5).
#
# Regression provenance remains Notebook 3's leak-free deployment selection.
# No algorithm is re-selected here from held-out test performance.
# =============================================================================
best_algo_df = DATA["best_algorithm"]

required_best_algo_cols = {"target", "activity_pool", "task", "best_algorithm"}
missing_best_algo_cols = required_best_algo_cols - set(best_algo_df.columns)
if missing_best_algo_cols:
    raise ValueError(
        "best_algorithm_by_combination.csv is missing required provenance columns: "
        f"{sorted(missing_best_algo_cols)}"
    )


def _unique_notebook3_algorithm(target, task="classification", pool=POOL):
    """Return one unambiguous Notebook 3 deployment label, or None if absent."""
    sub = best_algo_df[
        (best_algo_df["target"] == target)
        & (best_algo_df["activity_pool"] == pool)
        & (best_algo_df["task"] == task)
    ].copy()
    if len(sub) == 0:
        return None
    observed = sorted(sub["best_algorithm"].dropna().astype(str).str.strip().unique())
    if len(observed) != 1:
        raise ValueError(
            f"PROVENANCE FAILURE: {target} / {pool} / {task} has ambiguous "
            f"Notebook 3 deployment entries: {observed}. Notebook 7 will not "
            "choose one arbitrarily."
        )
    return observed[0]


# Recover classification deployment from Notebook 4's corrected primary screen.
PRIMARY_SCREEN_DEPLOYED_CLF = {}
PRIMARY_SCREEN_PROVENANCE_AVAILABLE = False
screening_candidates_for_provenance = DATA.get("screening_candidates")

if screening_candidates_for_provenance is not None:
    required_primary_cols = {"target", "activity_pool", "feature_representation", "algorithm"}
    missing_primary_cols = required_primary_cols - set(screening_candidates_for_provenance.columns)
    if missing_primary_cols:
        raise ValueError(
            "drugbank_primary_ranked_candidates.csv is present but missing required "
            f"provenance columns: {sorted(missing_primary_cols)}. Use the patched "
            "Notebook 4 primary artifact before running Notebook 7."
        )

    if not screening_candidates_for_provenance["activity_pool"].eq("full").all():
        raise ValueError(
            "PROVENANCE FAILURE: drugbank_primary_ranked_candidates.csv contains "
            "non-FULL activity-pool rows."
        )
    if not screening_candidates_for_provenance["feature_representation"].eq("combined").all():
        raise ValueError(
            "PROVENANCE FAILURE: drugbank_primary_ranked_candidates.csv contains "
            "non-COMBINED representation rows."
        )

    for target, grp in screening_candidates_for_provenance.groupby("target"):
        observed = sorted(grp["algorithm"].dropna().astype(str).str.strip().unique())
        if len(observed) != 1:
            raise ValueError(
                f"PROVENANCE FAILURE: Notebook 4 primary candidates for {target} "
                f"contain multiple algorithms: {observed}."
            )
        PRIMARY_SCREEN_DEPLOYED_CLF[str(target)] = observed[0]

    PRIMARY_SCREEN_PROVENANCE_AVAILABLE = bool(PRIMARY_SCREEN_DEPLOYED_CLF)


# Final classification map: Notebook 4 primary provenance, then Notebook 3 fallback.
DEPLOYED_CLF = {}
DEPLOYED_CLF_SOURCE = {}
for t in TARGETS:
    if t in PRIMARY_SCREEN_DEPLOYED_CLF:
        DEPLOYED_CLF[t] = PRIMARY_SCREEN_DEPLOYED_CLF[t]
        DEPLOYED_CLF_SOURCE[t] = "Notebook 4 primary FULL/combined screen"
    else:
        DEPLOYED_CLF[t] = _unique_notebook3_algorithm(t, "classification", POOL)
        DEPLOYED_CLF_SOURCE[t] = "Notebook 3 leak-free fallback"

# Regression is not part of Notebook 4's screening provenance.
DEPLOYED_REG = {t: _unique_notebook3_algorithm(t, "regression", POOL) for t in TARGETS}

missing_clf = [t for t in TARGETS if DEPLOYED_CLF[t] is None]
missing_reg = [t for t in TARGETS if DEPLOYED_REG[t] is None]
if missing_clf:
    raise ValueError(f"No deployed classification algorithm could be resolved for: {missing_clf}")
if missing_reg:
    raise ValueError(f"No deployed regression algorithm could be resolved for: {missing_reg}")

print("Deployed algorithm per target (provenance-preserving resolution):")
for t in TARGETS:
    print(
        f"  {TARGET_LABELS[t]:<8} classification={str(DEPLOYED_CLF[t]):<15} "
        f"[{DEPLOYED_CLF_SOURCE[t]}]  regression={DEPLOYED_REG[t]}"
    )

deployed_algo_df = pd.DataFrame([
    {
        "target": t,
        "deployed_algorithm_classification": DEPLOYED_CLF[t],
        "classification_source": DEPLOYED_CLF_SOURCE[t],
        "deployed_algorithm_regression": DEPLOYED_REG[t],
        "regression_source": "Notebook 3 leak-free selection",
        "headline_activity_pool": POOL,
        "headline_feature_representation": REPRESENTATION,
    }
    for t in TARGETS
])
save_table(deployed_algo_df, "deployed_algorithm_by_target")

# Audit local Notebook 3 classification table versus inherited production provenance.
notebook3_vs_production = []
for t in TARGETS:
    n3_algo = _unique_notebook3_algorithm(t, "classification", POOL)
    notebook3_vs_production.append({
        "target": t,
        "notebook3_recorded_classification": n3_algo,
        "production_classification_used": DEPLOYED_CLF[t],
        "production_source": DEPLOYED_CLF_SOURCE[t],
        "agree": str(n3_algo) == str(DEPLOYED_CLF[t]),
    })
notebook3_vs_production_df = pd.DataFrame(notebook3_vs_production)
save_table(notebook3_vs_production_df, "notebook3_vs_production_algorithm_provenance")

n_n3_disagree = int((~notebook3_vs_production_df["agree"]).sum())
if n_n3_disagree:
    print(
        f"\nNOTE: {n_n3_disagree} target(s) differ between the local Notebook 3 selection "
        "table and the classifier inherited from Notebook 4's corrected production screen. "
        "Notebook 7 uses the production-screen classifier for those classification analyses."
    )
    print(
        notebook3_vs_production_df.loc[
            ~notebook3_vs_production_df["agree"],
            ["target", "notebook3_recorded_classification", "production_classification_used", "production_source"]
        ].to_string(index=False)
    )

# Integrity check: Notebook 5 enrichment provenance must agree with resolved production model.
algo_disagreements = []
enrich_set = DATA.get("docking_enrichment_set")
if enrich_set is not None and {"target", "algorithm_used"} <= set(enrich_set.columns):
    for t, grp in enrich_set.groupby("target"):
        observed = sorted(grp["algorithm_used"].dropna().astype(str).str.strip().unique())
        if len(observed) != 1:
            algo_disagreements.append({
                "target": t,
                "expected_deployed_algorithm": DEPLOYED_CLF.get(t),
                "algorithm_used_by_docking_enrichment": ";".join(observed),
                "consequence": "ambiguous enrichment-model provenance; recompute before quoting ML-vs-docking correlation",
                "not_affected": "label-based enrichment metrics (ROC-AUC/BEDROC/EF) are unaffected",
            })
        elif t in DEPLOYED_CLF and observed[0] != str(DEPLOYED_CLF[t]):
            algo_disagreements.append({
                "target": t,
                "expected_deployed_algorithm": DEPLOYED_CLF[t],
                "algorithm_used_by_docking_enrichment": observed[0],
                "consequence": "ML-vs-docking correlation rows describe a non-production model",
                "not_affected": "label-based enrichment metrics (ROC-AUC/BEDROC/EF) are unaffected",
            })

algo_disagreement_columns = [
    "target", "expected_deployed_algorithm",
    "algorithm_used_by_docking_enrichment", "consequence", "not_affected",
]
algo_disagreement_df = pd.DataFrame(algo_disagreements, columns=algo_disagreement_columns)
save_table(algo_disagreement_df, "deployed_algorithm_disagreements")
ALGO_DISAGREEMENT_TARGETS = set(algo_disagreement_df["target"]) if len(algo_disagreement_df) else set()

if len(algo_disagreement_df):
    print("\n  WARNING: Notebook 5 enrichment provenance disagrees with the resolved production model:")
    print(algo_disagreement_df[[
        "target", "expected_deployed_algorithm", "algorithm_used_by_docking_enrichment"
    ]].to_string(index=False))
    print("  Affected ML-vs-docking correlation rows will be flagged in Module G.")
else:
    print("\nNotebook 5 enrichment provenance agrees with the resolved production classifiers.")


In [ ]:
# =============================================================================
# INTEGRITY CHECK 4 -- is Notebook 3's deployment table reproducible from the
# tuning history it claims to be derived from?
#
# This remains a hard integrity check on the local Notebook 3 results tree.
# Classification used by Notebook 7 may inherit a later corrected production
# choice from Notebook 4 (previous cell). Regression and any classification
# target absent from Notebook 4 still depend directly on this Notebook 3 table.
# =============================================================================
ALGO_KEY_TO_LABEL = {"rf": "Random Forest", "xgb": "XGBoost", "lgb": "LightGBM"}

selection_rows = []
for target in TARGETS:
    for pool in ["ki", "ki_ic50", "full"]:
        history_path = results_path / f"{target}_{pool}" / "optuna_tuning_history.json"
        if not history_path.exists():
            continue
        with open(history_path) as fh:
            history = json.load(fh)
        for task in ["classification", "regression"]:
            values = {
                ALGO_KEY_TO_LABEL[k.split("_")[0]]: float(v["best_value"])
                for k, v in history.items()
                if k.endswith(task)
                and k.split("_")[0] in ALGO_KEY_TO_LABEL
                and isinstance(v, dict)
                and "best_value" in v
            }
            if not values:
                continue
            recomputed = max(values, key=values.get)
            row = best_algo_df[
                (best_algo_df["target"] == target)
                & (best_algo_df["activity_pool"] == pool)
                & (best_algo_df["task"] == task)
            ]
            recorded_values = sorted(
                row["best_algorithm"].dropna().astype(str).str.strip().unique()
            ) if len(row) else []
            recorded = recorded_values[0] if len(recorded_values) == 1 else None
            selection_rows.append({
                "target": target,
                "activity_pool": pool,
                "task": task,
                "recorded": recorded,
                "recorded_values": ";".join(recorded_values),
                "recomputed_from_tuning_history": recomputed,
                "agree": recorded == recomputed,
                **{f"best_value_{k}": v for k, v in values.items()},
            })

selection_check_df = pd.DataFrame(selection_rows)
save_table(selection_check_df, "deployed_algorithm_selection_check")

if len(selection_check_df):
    disagreements = selection_check_df[~selection_check_df["agree"]]
    print(
        "Notebook 3 selection table vs its local tuning history: "
        f"{int(selection_check_df['agree'].sum())}/{len(selection_check_df)} agree"
    )
    if len(disagreements):
        print()
        print(disagreements[[
            "target", "activity_pool", "task", "recorded_values",
            "recomputed_from_tuning_history"
        ]].to_string(index=False))
        raise ValueError(
            "The local Notebook 3 deployed-algorithm table is not reproducible from its own "
            "tuning history. This indicates a mixed or stale Notebook 3 results tree. Resolve "
            "the Notebook 3 artifacts before assembling the meta-analysis."
        )
    print("Notebook 3 selection-table integrity check passed.")
else:
    print("No optuna_tuning_history.json files found; Notebook 3 selection-table check skipped.")

SELECTION_TABLE_SHA256 = _sha256_file(results_path / "best_algorithm_by_combination.csv")
print(f"Notebook 3 selection table SHA-256: {SELECTION_TABLE_SHA256}")


## Module B: Model Performance Landscape

One aligned per-target table carrying internal and external evidence together,
with the evidence strength of each external row made explicit so a 6-compound
BindingDB result can never be read as equal in weight to a 300-compound temporal
result.

Columns are grouped as: deployed model, internal scaffold-split performance,
reliability (calibration, conformal, applicability domain), and external
validation on both independent axes.

In [ ]:
# =============================================================================
# MODULE B -- cross-target performance landscape.
# =============================================================================
final_perf = DATA["final_performance"]
calib = DATA["calibration"]
conf = DATA["conformal"]
preds = DATA["test_predictions"]
extval = DATA["external_validation"]

def _first(df, **filters):
    """Return the first matching row as a dict, or {} if nothing matches. Used
    everywhere below so a missing upstream row yields NaN in one cell rather
    than raising and losing the whole table."""
    sub = df
    for col, val in filters.items():
        if col not in sub.columns:
            return {}
        sub = sub[sub[col] == val]
    return {} if len(sub) == 0 else sub.iloc[0].to_dict()

landscape_records = []
for t in TARGETS:
    algo_c, algo_r = DEPLOYED_CLF[t], DEPLOYED_REG[t]

    perf_c = _first(final_perf, target=t, activity_pool=POOL,
                    feature_representation=REPRESENTATION, algorithm=algo_c)
    perf_r = _first(final_perf, target=t, activity_pool=POOL,
                    feature_representation=REPRESENTATION, algorithm=algo_r)
    cal = _first(calib, target=t, activity_pool=POOL,
                 feature_representation=REPRESENTATION, algorithm=algo_c)
    conf_c = _first(conf, target=t, activity_pool=POOL, feature_representation=REPRESENTATION,
                    algorithm=algo_c, task="classification",
                    confidence_level=CONFIDENCE_LEVEL, stratum="overall")
    conf_r = _first(conf, target=t, activity_pool=POOL, feature_representation=REPRESENTATION,
                    algorithm=algo_r, task="regression",
                    confidence_level=CONFIDENCE_LEVEL, stratum="overall")

    pr = preds[(preds["target"] == t) & (preds["activity_pool"] == POOL)
               & (preds["feature_representation"] == REPRESENTATION)
               & (preds["algorithm"] == algo_c)]
    pct_ad = (100.0 * pr["within_applicability_domain"].astype(bool).mean()) if len(pr) else np.nan

    bdb = _first(extval, target=t, axis="bindingdb")
    temporal_rows = extval[(extval["target"] == t) & (extval["axis"] == "temporal")]
    never_seen = temporal_rows[temporal_rows["stratum"].astype(str).str.contains("never_seen", na=False)]
    tmp = never_seen.iloc[0].to_dict() if len(never_seen) else (
        temporal_rows.iloc[0].to_dict() if len(temporal_rows) else {})

    def _strength(n):
        if n is None or (isinstance(n, float) and np.isnan(n)):
            return "unavailable"
        return "primary" if n >= MIN_N_FOR_PRIMARY_CLAIM else "weak (small n)"

    landscape_records.append({
        "target": TARGET_LABELS[t],
        "target_key": t,
        "deployed_algorithm_classification": algo_c,
        "deployed_algorithm_regression": algo_r,
        "internal_test_roc_auc": perf_c.get("test_roc_auc", np.nan),
        "internal_test_r2": perf_r.get("test_r2", np.nan),
        "brier_raw": cal.get("brier_raw", np.nan),
        "brier_calibrated": cal.get("brier_calibrated", np.nan),
        "ece_raw": cal.get("ece_raw", np.nan),
        "ece_calibrated": cal.get("ece_calibrated", np.nan),
        "venn_abers_mean_interval_width": cal.get("venn_abers_mean_interval_width", np.nan),
        "conformal_coverage_classification": conf_c.get("empirical_coverage", np.nan),
        "conformal_mean_set_size": conf_c.get("mean_set_size", np.nan),
        "conformal_coverage_regression": conf_r.get("empirical_coverage", np.nan),
        "conformal_mean_interval_width": conf_r.get("mean_interval_width", np.nan),
        "pct_test_within_applicability_domain": pct_ad,
        "bindingdb_n": bdb.get("n", np.nan),
        "bindingdb_roc_auc": bdb.get("roc_auc", np.nan),
        "bindingdb_evidence_strength": _strength(bdb.get("n", np.nan)),
        "temporal_stratum": tmp.get("stratum", ""),
        "temporal_n": tmp.get("n", np.nan),
        "temporal_roc_auc": tmp.get("roc_auc", np.nan),
        "temporal_evidence_strength": _strength(tmp.get("n", np.nan)),
        "conformal_nominal_level": CONFIDENCE_LEVEL,
    })

landscape_df = pd.DataFrame(landscape_records)
save_table(landscape_df, "model_performance_landscape")

display_cols = ["target", "deployed_algorithm_classification", "internal_test_roc_auc",
                "internal_test_r2", "brier_calibrated", "ece_calibrated",
                "conformal_coverage_classification", "pct_test_within_applicability_domain",
                "temporal_n", "temporal_roc_auc", "bindingdb_n", "bindingdb_roc_auc",
                "bindingdb_evidence_strength"]
print()
print(landscape_df[display_cols].round(3).to_string(index=False))

# Consistency statement across targets -- the answer to core question 1 is a
# spread, not a single number, so the spread itself is persisted.
spread_records = []
for metric in ["internal_test_roc_auc", "internal_test_r2", "brier_calibrated",
               "ece_calibrated", "conformal_coverage_classification",
               "pct_test_within_applicability_domain", "temporal_roc_auc"]:
    v = pd.to_numeric(landscape_df[metric], errors="coerce").dropna()
    if len(v) == 0:
        continue
    spread_records.append({
        "metric": metric, "n_targets": len(v), "min": v.min(), "max": v.max(),
        "range": v.max() - v.min(), "mean": v.mean(), "sd": v.std(ddof=1) if len(v) > 1 else np.nan,
        "coefficient_of_variation": (v.std(ddof=1) / v.mean()) if len(v) > 1 and v.mean() else np.nan,
        "best_target": landscape_df.loc[v.idxmax(), "target"],
        "worst_target": landscape_df.loc[v.idxmin(), "target"],
    })

consistency_df = pd.DataFrame(spread_records)
save_table(consistency_df, "cross_target_consistency_spread")
print()
print(consistency_df.round(4).to_string(index=False))

## Module C: Target Difficulty Drivers and Dataset Correlates

Assembles one row per target of the dataset properties characterized in notebook
2 -- size, class balance, scaffold diversity, singleton fraction, assay support,
endpoint composition, measurement density, structural-alert burden -- and relates
them to the performance metrics from Module B.

**Interpretive limit, stated in the output itself rather than left to the
reader:** these are rank correlations over five targets. A Spearman rho computed
on n = 5 can only take a small number of discrete values and its p-value has no
useful resolution. The correlations are saved with their n and an explicit
`interpretation` column marking them descriptive; they order hypotheses, they do
not test them.

In [ ]:
# =============================================================================
# MODULE C -- target-level properties vs performance.
# =============================================================================
ds = DATA["dataset_summary"]
ds_full = ds[ds["activity_pool"] == POOL].copy()

DATASET_PROPERTY_COLS = [
    "final_unique_compounds", "raw_records", "active_pct", "n_scaffolds",
    "scaffold_richness", "singleton_scaffold_pct", "largest_scaffold_pct",
    "median_scaffold_size", "single_measurement_pct", "single_assay_pct",
    "multi_activity_type_pct", "measurements_median", "assays_median",
    "pActivity_sd", "pActivity_iqr", "iqr_outlier_pct", "pains_free_pct",
    "brenk_free_pct", "any_alert_free_pct", "lipinski_compliant_pct",
]
keep = ["target"] + [c for c in DATASET_PROPERTY_COLS if c in ds_full.columns]
props = ds_full[keep].copy()

# ---- scaffold diversity (notebook 2) ----
sd = DATA["scaffold_diversity"]
sd_full = sd[sd["activity_pool"] == POOL]
sd_cols = [c for c in ["avg_compounds_per_scaffold", "pct_singleton_scaffolds",
                       "top_5_scaffold_coverage_pct", "top_10_scaffold_coverage_pct",
                       "median_compounds_per_scaffold"] if c in sd_full.columns]
props = props.merge(sd_full[["target"] + sd_cols], on="target", how="left")

# ---- endpoint composition (assay heterogeneity proxy) ----
ec = DATA["endpoint_composition"]
ec_full = ec[ec["activity_pool"] == POOL]
ec_cols = [c for c in ["pct_retained_ki_measurements", "pct_retained_ic50_measurements",
                       "pct_retained_ec50_measurements", "pct_compounds_multi_endpoint_type"]
           if c in ec_full.columns]
props = props.merge(ec_full[["target"] + ec_cols], on="target", how="left")

# ---- assay support distribution -> one column per support category ----
asup = DATA["assay_support"]
asup_full = asup[asup["activity_pool"] == POOL]
if len(asup_full):
    asup_wide = (asup_full.pivot_table(index="target", columns="support_category",
                                       values="pct", aggfunc="first")
                 .add_prefix("assay_support_pct_").reset_index())
    props = props.merge(asup_wide, on="target", how="left")

# ---- measurement-density distribution -> share of single-measurement compounds ----
msup = DATA.get("measurement_support")
if msup is not None:
    msup_full = msup[msup["activity_pool"] == POOL]
    single_bin = msup_full[msup_full["measurements_bin"].astype(str) == "1"]
    if len(single_bin):
        props = props.merge(
            single_bin[["target", "pct"]].rename(columns={"pct": "pct_single_measurement_compounds"}),
            on="target", how="left")

# ---- structural alerts ----
sa = DATA.get("structural_alerts")
if sa is not None:
    sa_full = sa[sa["activity_pool"] == POOL]
    sa_cols = [c for c in ["pains_flagged_pct", "brenk_flagged_pct", "any_alert_pct"]
               if c in sa_full.columns]
    if sa_cols:
        props = props.merge(sa_full[["target"] + sa_cols], on="target", how="left")

# ---- attach performance ----
perf_cols = ["target_key", "internal_test_roc_auc", "internal_test_r2", "ece_calibrated",
             "brier_calibrated", "conformal_coverage_classification",
             "pct_test_within_applicability_domain", "temporal_roc_auc", "temporal_n",
             "bindingdb_roc_auc"]
props = props.merge(landscape_df[perf_cols].rename(columns={"target_key": "target"}),
                    on="target", how="left")
props["target_label"] = props["target"].map(TARGET_LABELS)

target_properties_df = props
save_table(target_properties_df, "target_properties_vs_performance")
print()
print(target_properties_df[["target_label", "final_unique_compounds", "active_pct",
                            "pct_singleton_scaffolds", "internal_test_roc_auc",
                            "temporal_roc_auc"]].round(3).to_string(index=False))

In [ ]:
# =============================================================================
# MODULE C -- rank correlations, explicitly labelled descriptive at n = 5.
# =============================================================================
PERFORMANCE_METRICS = ["internal_test_roc_auc", "internal_test_r2", "temporal_roc_auc",
                       "ece_calibrated", "conformal_coverage_classification"]
# Several notebook 2 columns are exact complements of each other (a "% flagged"
# and its "% free" counterpart sum to 100). Keeping both would report the same
# association twice with opposite signs and pad the ranked table with duplicates,
# so one of each complementary pair is dropped here rather than at reading time.
COMPLEMENTARY_DROP = ["pains_free_pct", "brenk_free_pct", "any_alert_free_pct",
                      "inactive_pct", "pct_non_singleton_scaffolds"]

property_cols = [c for c in target_properties_df.columns
                 if c not in PERFORMANCE_METRICS + COMPLEMENTARY_DROP
                 + ["target", "target_label", "target_key", "temporal_n",
                    "bindingdb_roc_auc", "brier_calibrated",
                    "pct_test_within_applicability_domain"]]

corr_records = []
for prop in property_cols:
    x = pd.to_numeric(target_properties_df[prop], errors="coerce")
    if x.notna().sum() < 3 or x.nunique(dropna=True) < 3:
        continue
    for metric in PERFORMANCE_METRICS:
        y = pd.to_numeric(target_properties_df[metric], errors="coerce")
        mask = x.notna() & y.notna()
        if mask.sum() < 3:
            continue
        rho, p = stats.spearmanr(x[mask], y[mask])
        corr_records.append({
            "property": prop, "performance_metric": metric, "n_targets": int(mask.sum()),
            "spearman_rho": rho, "spearman_p": p,
            "abs_rho": abs(rho) if rho == rho else np.nan,
            "interpretation": ("descriptive only -- n=5 targets, p-value has no useful "
                               "resolution and must not be read as a significance test"),
        })

driver_corr_df = (pd.DataFrame(corr_records)
                  .sort_values(["abs_rho", "property"], ascending=[False, True],
                               ignore_index=True))

# With five targets a Spearman rho can only take a handful of discrete values, so
# many properties tie at the top and "the strongest predictor" is a tie-break, not
# a finding. The size of each tie group is recorded so no single row can be quoted
# as if it stood alone.
if len(driver_corr_df):
    tie_sizes = driver_corr_df.groupby(driver_corr_df["spearman_rho"].round(6))["property"].transform("size")
    driver_corr_df["n_properties_tied_at_this_rho"] = tie_sizes
save_table(driver_corr_df, "performance_driver_correlations")

print()
print("Strongest property/performance rank associations (descriptive, n=5 targets):")
print(driver_corr_df.head(12)[["property", "performance_metric", "spearman_rho", "n_targets",
                               "n_properties_tied_at_this_rho"]].round(3).to_string(index=False))
if len(driver_corr_df):
    top_rho = driver_corr_df.iloc[0]["spearman_rho"]
    n_tied = int(driver_corr_df.iloc[0]["n_properties_tied_at_this_rho"])
    if n_tied > 1:
        print(f"\n  {n_tied} property/metric pairs tie at rho = {top_rho:.2f}. At five targets the "
              f"rank correlation is coarse enough that ordering within a tie carries no "
              f"information: report the tied set, not a single winner.")

# ---- The size-does-not-explain-performance observation, computed not asserted ----
size_perf = target_properties_df[["target_label", "final_unique_compounds",
                                  "internal_test_roc_auc"]].dropna().copy()
size_perf["dataset_size_rank"] = size_perf["final_unique_compounds"].rank(ascending=False).astype(int)
size_perf["roc_auc_rank"] = size_perf["internal_test_roc_auc"].rank(ascending=False).astype(int)
size_perf["rank_discordance"] = size_perf["dataset_size_rank"] - size_perf["roc_auc_rank"]
save_table(size_perf, "dataset_size_vs_performance_ranks")

largest = size_perf.loc[size_perf["dataset_size_rank"].idxmin()]
best = size_perf.loc[size_perf["roc_auc_rank"].idxmin()]
rho_size, p_size = stats.spearmanr(size_perf["final_unique_compounds"],
                                   size_perf["internal_test_roc_auc"])
print()
print(size_perf.round(3).to_string(index=False))
print(f"\nLargest dataset: {largest['target_label']} "
      f"({int(largest['final_unique_compounds'])} compounds, "
      f"test ROC-AUC {largest['internal_test_roc_auc']:.3f})")
print(f"Best test ROC-AUC: {best['target_label']} ({best['internal_test_roc_auc']:.3f}, "
      f"{int(best['final_unique_compounds'])} compounds)")
print(f"Spearman(dataset size, test ROC-AUC) = {rho_size:.3f} (n=5, descriptive only)")
if largest["target_label"] != best["target_label"]:
    print("Dataset size alone does not determine performance in this panel: the largest "
          "dataset is not the best-performing target.")

## Module D: Feature Representation and SHAP Synthesis

Two questions, in order. First the coarse one: does the combined representation
actually beat Morgan fingerprints or physicochemical descriptors alone, and is
that answer the same for every receptor? Then the fine one: which individual
features carry importance across targets versus only in one.

SHAP shares are normalized within each (target, representation) before any
cross-target comparison. Raw mean absolute SHAP values are on different scales
per model and per target, so comparing them directly would rank targets by model
output scale rather than by feature importance.

In [ ]:
# =============================================================================
# MODULE D (part 1) -- feature representation comparison.
# =============================================================================
FEATURE_REPS = ["morgan", "descriptors", "combined"]
FEATURE_REP_LABELS = {"morgan": "Morgan fingerprints",
                      "descriptors": "Physicochemical descriptors",
                      "combined": "Morgan + descriptors"}

rep_records = []
for t in TARGETS:
    algo_c, algo_r = DEPLOYED_CLF[t], DEPLOYED_REG[t]
    for rep in FEATURE_REPS:
        row_c = _first(final_perf, target=t, activity_pool=POOL,
                       feature_representation=rep, algorithm=algo_c)
        row_r = _first(final_perf, target=t, activity_pool=POOL,
                       feature_representation=rep, algorithm=algo_r)
        sub_any = final_perf[(final_perf["target"] == t)
                             & (final_perf["activity_pool"] == POOL)
                             & (final_perf["feature_representation"] == rep)]
        rep_records.append({
            "target": TARGET_LABELS[t], "target_key": t,
            "feature_representation": rep,
            "feature_representation_label": FEATURE_REP_LABELS[rep],
            "deployed_algorithm_classification": algo_c,
            "test_roc_auc_deployed_algorithm": row_c.get("test_roc_auc", np.nan),
            "test_r2_deployed_algorithm": row_r.get("test_r2", np.nan),
            "test_roc_auc_best_any_algorithm": (sub_any["test_roc_auc"].max()
                                                if len(sub_any) else np.nan),
        })

representation_df = pd.DataFrame(rep_records)
save_table(representation_df, "feature_representation_comparison")

wide = representation_df.pivot_table(index=["target", "target_key"],
                                     columns="feature_representation",
                                     values="test_roc_auc_deployed_algorithm").reset_index()
for rep in ("morgan", "descriptors"):
    if rep in wide.columns and "combined" in wide.columns:
        wide[f"combined_minus_{rep}"] = wide["combined"] - wide[rep]
if {"morgan", "descriptors"} <= set(wide.columns):
    wide["best_single_representation"] = np.where(wide["morgan"] >= wide["descriptors"],
                                                  "Morgan fingerprints",
                                                  "Physicochemical descriptors")
    wide["combined_is_best"] = wide["combined"] >= wide[["morgan", "descriptors"]].max(axis=1)

representation_wide_df = wide
save_table(representation_wide_df, "feature_representation_comparison_wide")
print()
print(representation_wide_df.round(4).to_string(index=False))

if "combined_is_best" in representation_wide_df.columns:
    n_best = int(representation_wide_df["combined_is_best"].sum())
    print(f"\nCombined representation is best (or tied) for {n_best}/"
          f"{len(representation_wide_df)} targets at the deployed algorithm.")

In [ ]:
# =============================================================================
# MODULE D (part 2) -- SHAP synthesis across targets.
#
# Shares are normalized within each (target, representation): raw mean absolute
# SHAP values live on different scales per model, so an unnormalized
# cross-target comparison would rank targets by output scale, not by which
# feature matters.
# =============================================================================
DESCRIPTOR_COLS = ["MW", "LogP", "TPSA", "HBD", "HBA", "RotBonds",
                   "HeavyAtomCount", "RingCount", "AromaticRingCount", "FractionCSP3"]

shap_entries = [e for e in DATA["shap_summaries"]
                if e.get("activity_pool") == POOL and e.get("target") in TARGETS]

shap_rows = []
for e in shap_entries:
    feats = e.get("top_features", []) or []
    vals = e.get("mean_abs_shap", []) or []
    total = float(np.sum(vals)) if len(vals) else 0.0
    for rank, (f, v) in enumerate(zip(feats, vals), start=1):
        shap_rows.append({
            "target": e["target"], "target_label": TARGET_LABELS.get(e["target"], e["target"]),
            "feature_representation": e.get("feature_representation"),
            "algorithm": e.get("algorithm"), "feature": f, "rank_within_model": rank,
            "mean_abs_shap": float(v),
            "share_of_top_features": (float(v) / total) if total else np.nan,
            "feature_type": "descriptor" if f in DESCRIPTOR_COLS else "Morgan bit",
        })

shap_long_df = pd.DataFrame(shap_rows)
save_table(shap_long_df, "shap_feature_shares_long")

if len(shap_long_df):
    print()
    print("SHAP entries loaded per representation (full pool):")
    print(shap_long_df.groupby(["feature_representation", "target_label"]).size()
          .unstack(fill_value=0).to_string())

    # ---- descriptor heatmap: descriptors x targets, combined representation ----
    for rep in ["combined", "descriptors"]:
        sub = shap_long_df[(shap_long_df["feature_representation"] == rep)
                           & (shap_long_df["feature_type"] == "descriptor")]
        if not len(sub):
            continue
        heat = (sub.pivot_table(index="feature", columns="target_label",
                                values="share_of_top_features", aggfunc="first")
                .reindex([d for d in DESCRIPTOR_COLS if d in set(sub["feature"])])
                .reindex(columns=[TARGET_LABELS[t] for t in TARGETS if TARGET_LABELS[t]
                                  in set(sub["target_label"])]))
        heat_out = heat.reset_index()
        save_table(heat_out, f"shap_descriptor_heatmap_{rep}")
        if rep == "combined":
            shap_heatmap_combined = heat

    # ---- transferability: in how many targets does a feature reach the top list ----
    # How many features does each representation actually list per target? SHAP
    # summaries store a truncated top-N list, so "appears in N targets" is a
    # statement about the truncated list, not about the whole feature space. When
    # the listed length equals the representation's entire feature set (the
    # descriptor-only models list all 10 descriptors), every feature trivially
    # appears in every target and the count carries no information -- only the
    # share does. That distinction is written into the table rather than left for
    # a reader to infer.
    list_len = (shap_long_df.groupby(["feature_representation", "target"])["feature"]
                .count().groupby("feature_representation").max())
    feature_space = (shap_long_df.groupby("feature_representation")["feature"].nunique())

    trans = (shap_long_df.groupby(["feature_representation", "feature", "feature_type"])
             .agg(n_targets_in_top_list=("target", "nunique"),
                  mean_share=("share_of_top_features", "mean"),
                  mean_rank=("rank_within_model", "mean"),
                  best_rank=("rank_within_model", "min"),
                  targets=("target_label", lambda s: ",".join(sorted(set(s)))))
             .reset_index())
    trans["features_listed_per_target"] = trans["feature_representation"].map(list_len)
    trans["distinct_features_observed"] = trans["feature_representation"].map(feature_space)
    trans["count_is_informative"] = (trans["distinct_features_observed"]
                                     > trans["features_listed_per_target"])
    trans["count_caveat"] = np.where(
        trans["count_is_informative"], "",
        "every feature of this representation is listed for every target, so the target count is "
        "an artifact of the list length; compare mean_share instead")
    trans = trans.sort_values(["feature_representation", "n_targets_in_top_list", "mean_share"],
                              ascending=[True, False, False], ignore_index=True)
    feature_transferability_df = trans
    save_table(feature_transferability_df, "feature_transferability_summary")

    n_listed_combined = int(list_len.get("combined", np.nan))
    print()
    print(f"Combined-representation models list the top {n_listed_combined} features each, so the "
          f"counts below mean 'reaches the top {n_listed_combined}', not 'is unimportant elsewhere'.")
    print(f"Features reaching that list in all {len(TARGETS)} targets:")
    comb_trans = trans[trans["feature_representation"] == "combined"]
    universal = comb_trans[comb_trans["n_targets_in_top_list"] == len(TARGETS)]
    if len(universal):
        print(universal[["feature", "feature_type", "mean_share", "best_rank"]]
              .round(4).to_string(index=False))
    else:
        print("  none -- no single feature reaches the truncated top list in every target")
        near = comb_trans[comb_trans["n_targets_in_top_list"] == len(TARGETS) - 1]
        if len(near):
            print(f"  most widely shared ({len(TARGETS) - 1} of {len(TARGETS)} targets):")
            print(near[["feature", "feature_type", "n_targets_in_top_list", "mean_share",
                        "targets"]].round(4).to_string(index=False))

    n_by_type = (trans[trans["feature_representation"] == "combined"]
                 .groupby("feature_type")["n_targets_in_top_list"]
                 .agg(["count", "mean"]).reset_index()
                 .rename(columns={"count": "n_features", "mean": "mean_targets_shared"}))
    save_table(n_by_type, "feature_transferability_by_type")
    print()
    print(n_by_type.round(3).to_string(index=False))
else:
    print("No SHAP entries matched the headline pool; Module D part 2 produced no output.")

## Module E: Reliability and Uncertainty Synthesis

The methodological heart of the paper: whether the uncertainty machinery behaves
the same way on five different receptors. Four pieces, each computed across all
targets and saved separately:

1. **Calibration** -- raw versus calibrated Brier and expected calibration error,
   so the improvement attributable to calibration is visible rather than implied.
2. **Conformal coverage** -- empirical coverage against the nominal level, split
   by applicability-domain status and by scaffold singleton status.
3. **Applicability domain** -- does error actually rise outside the domain, and
   does the domain distance track error magnitude.
4. **Confidence stratification** -- are high-confidence predictions consistently
   more accurate, on every target.

**One structural caveat is tested rather than assumed.** Split conformal
regression without a normalizing difficulty estimate produces a constant interval
width for every compound, so "intervals widen outside the applicability domain"
would be untestable by construction. The cell below measures whether width
actually varies by stratum and reports which regime this pipeline is in, so the
manuscript makes the claim that the data supports rather than the one the
framework is usually assumed to support.

In [ ]:
# =============================================================================
# MODULE E (part 1) -- calibration across targets.
# =============================================================================
calib_records = []
for t in TARGETS:
    row = _first(calib, target=t, activity_pool=POOL,
                 feature_representation=REPRESENTATION, algorithm=DEPLOYED_CLF[t])
    if not row:
        continue
    brier_raw, brier_cal = row.get("brier_raw", np.nan), row.get("brier_calibrated", np.nan)
    ece_raw, ece_cal = row.get("ece_raw", np.nan), row.get("ece_calibrated", np.nan)
    calib_records.append({
        "target": TARGET_LABELS[t], "target_key": t, "algorithm": DEPLOYED_CLF[t],
        "brier_raw": brier_raw, "brier_calibrated": brier_cal,
        "brier_improvement": brier_raw - brier_cal,
        "brier_improvement_pct": (100.0 * (brier_raw - brier_cal) / brier_raw)
                                 if brier_raw else np.nan,
        "ece_raw": ece_raw, "ece_calibrated": ece_cal,
        "ece_improvement": ece_raw - ece_cal,
        "calibration_slope_calibrated": row.get("calibration_slope_calibrated", np.nan),
        "calibration_intercept_calibrated": row.get("calibration_intercept_calibrated", np.nan),
        "venn_abers_mean_interval_width": row.get("venn_abers_mean_interval_width", np.nan),
        "venn_abers_std_interval_width": row.get("venn_abers_std_interval_width", np.nan),
    })

calibration_cross_target_df = pd.DataFrame(calib_records)
save_table(calibration_cross_target_df, "calibration_across_targets")
print()
print(calibration_cross_target_df.round(4).to_string(index=False))

if len(calibration_cross_target_df):
    n_t = len(calibration_cross_target_df)
    n_ece_better = int((calibration_cross_target_df["ece_improvement"] > 0).sum())
    n_brier_better = int((calibration_cross_target_df["brier_improvement"] > 0).sum())
    print(f"\nCalibration lowered expected calibration error for {n_ece_better}/{n_t} targets "
          f"and lowered the Brier score for {n_brier_better}/{n_t}.")
    if n_ece_better < n_t or n_brier_better < n_t:
        worst = calibration_cross_target_df.loc[
            calibration_cross_target_df["ece_improvement"].idxmin()]
        print(f"  Calibration did NOT improve these metrics everywhere: the largest degradation is "
              f"{worst['target']} (expected calibration error {worst['ece_raw']:.4f} raw -> "
              f"{worst['ece_calibrated']:.4f} calibrated).")
        print(f"  Report this direction as measured. A calibration layer that trades sharpness for "
              f"distribution-free validity can raise these summary scores while still supplying the "
              f"probability intervals the pipeline relies on; what it must not do is get described "
              f"as an improvement the held-out data does not show.")

In [ ]:
# =============================================================================
# MODULE E (part 2) -- conformal coverage by stratum, both tasks.
# =============================================================================
conf_records = []
for t in TARGETS:
    for task, algo in (("classification", DEPLOYED_CLF[t]), ("regression", DEPLOYED_REG[t])):
        sub = conf[(conf["target"] == t) & (conf["activity_pool"] == POOL)
                   & (conf["feature_representation"] == REPRESENTATION)
                   & (conf["algorithm"] == algo) & (conf["task"] == task)]
        for _, r in sub.iterrows():
            nominal = float(r["confidence_level"])
            cov = float(r["empirical_coverage"]) if pd.notna(r["empirical_coverage"]) else np.nan
            conf_records.append({
                "target": TARGET_LABELS[t], "target_key": t, "task": task,
                "algorithm": algo, "nominal_level": nominal, "stratum": r["stratum"],
                "n": r.get("n", np.nan), "empirical_coverage": cov,
                "coverage_minus_nominal": cov - nominal,
                "covers_nominal": bool(cov >= nominal) if cov == cov else False,
                "mean_set_size": r.get("mean_set_size", np.nan),
                "mean_interval_width": r.get("mean_interval_width", np.nan),
            })

conformal_cross_target_df = pd.DataFrame(conf_records)
save_table(conformal_cross_target_df, "conformal_coverage_across_targets")

headline_conf = conformal_cross_target_df[
    (conformal_cross_target_df["nominal_level"] == CONFIDENCE_LEVEL)
    & (conformal_cross_target_df["stratum"].isin(["overall", "within_ad", "outside_ad",
                                                  "singleton_scaffold", "non_singleton_scaffold"]))]
print()
print(headline_conf.pivot_table(index=["task", "target"], columns="stratum",
                                values="empirical_coverage").round(3).to_string())

# ---- Is the AD effect visible in coverage, in width, or in neither? ----
width_records = []
for task in ("classification", "regression"):
    sub = conformal_cross_target_df[(conformal_cross_target_df["task"] == task)
                                    & (conformal_cross_target_df["nominal_level"] == CONFIDENCE_LEVEL)]
    piv = sub.pivot_table(index="target", columns="stratum",
                          values=["empirical_coverage", "mean_interval_width", "mean_set_size"])
    for t in piv.index:
        rec = {"task": task, "target": t}
        for metric in ("empirical_coverage", "mean_interval_width", "mean_set_size"):
            for stratum in ("within_ad", "outside_ad"):
                key = (metric, stratum)
                rec[f"{metric}_{stratum}"] = piv.loc[t, key] if key in piv.columns else np.nan
        rec["coverage_drop_outside_ad"] = (rec.get("empirical_coverage_within_ad", np.nan)
                                           - rec.get("empirical_coverage_outside_ad", np.nan))
        rec["width_increase_outside_ad"] = (rec.get("mean_interval_width_outside_ad", np.nan)
                                            - rec.get("mean_interval_width_within_ad", np.nan))
        rec["set_size_increase_outside_ad"] = (rec.get("mean_set_size_outside_ad", np.nan)
                                               - rec.get("mean_set_size_within_ad", np.nan))
        width_records.append(rec)

ad_conformal_df = pd.DataFrame(width_records)
save_table(ad_conformal_df, "conformal_ad_stratified_effects")
print()
print(ad_conformal_df.round(4).to_string(index=False))

reg = ad_conformal_df[ad_conformal_df["task"] == "regression"]
if len(reg) and reg["width_increase_outside_ad"].notna().any():
    # Relative, not absolute, tolerance: interval widths are ~2-3 pActivity units
    # and round-tripping through CSV leaves differences of order 1e-7 that are
    # storage artifacts, not real variation. An absolute epsilon would misread
    # those as evidence that width responds to domain status.
    max_abs_width_change = reg["width_increase_outside_ad"].abs().max()
    typical_width = reg["mean_interval_width_within_ad"].abs().mean()
    relative_width_change = (max_abs_width_change / typical_width
                             if typical_width and typical_width == typical_width else np.inf)
    if relative_width_change < 1e-4:
        print(f"\nRegression interval width is constant within and outside the applicability "
              f"domain for every target (largest difference {max_abs_width_change:.2g}, "
              f"{100 * relative_width_change:.2g}% of typical width -- a storage rounding artifact, "
              f"not a real effect). That is the expected behavior of split conformal "
              "regression without a normalizing difficulty estimate -- the interval is a single "
              "global quantile, so width cannot respond to domain status by construction. "
              "The testable claim for this pipeline is therefore about COVERAGE across strata, "
              "not about interval widening; report it that way rather than claiming widening "
              "the method cannot produce.")
    else:
        print(f"\nRegression interval width does vary by domain status (largest change "
              f"{max_abs_width_change:.4g}, {100 * relative_width_change:.3g}% of typical width); "
              f"widening outside the applicability domain is a testable claim for this pipeline.")

In [ ]:
# =============================================================================
# MODULE E (part 3) -- applicability domain vs error, and confidence
# stratification, computed per target from the deployed model's test
# predictions.
# =============================================================================
CONFIDENCE_BINS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

ad_error_records, confidence_records = [], []
for t in TARGETS:
    pr = preds[(preds["target"] == t) & (preds["activity_pool"] == POOL)
               & (preds["feature_representation"] == REPRESENTATION)
               & (preds["algorithm"] == DEPLOYED_CLF[t])].copy()
    if not len(pr):
        continue
    pr["within_ad"] = pr["within_applicability_domain"].astype(bool)
    pr["correct"] = (pr["y_true_class"] == pr["predicted_class"])

    rec = {"target": TARGET_LABELS[t], "target_key": t, "n_test": len(pr),
           "pct_within_ad": 100.0 * pr["within_ad"].mean()}
    for label, mask in (("within_ad", pr["within_ad"]), ("outside_ad", ~pr["within_ad"])):
        rec[f"n_{label}"] = int(mask.sum())
        rec[f"error_rate_{label}"] = (100.0 * (1 - pr.loc[mask, "correct"].mean())
                                      if mask.sum() else np.nan)
    rec["error_rate_increase_outside_ad"] = rec["error_rate_outside_ad"] - rec["error_rate_within_ad"]

    if {"y_true_pactivity", "predicted_pactivity"} <= set(pr.columns):
        pr["abs_error"] = (pr["y_true_pactivity"] - pr["predicted_pactivity"]).abs()
        for label, mask in (("within_ad", pr["within_ad"]), ("outside_ad", ~pr["within_ad"])):
            rec[f"mean_abs_error_{label}"] = (pr.loc[mask, "abs_error"].mean()
                                              if mask.sum() else np.nan)
        rec["mean_abs_error_increase_outside_ad"] = (rec.get("mean_abs_error_outside_ad", np.nan)
                                                     - rec.get("mean_abs_error_within_ad", np.nan))
        if "applicability_domain_distance" in pr.columns:
            m = pr["abs_error"].notna() & pr["applicability_domain_distance"].notna()
            if m.sum() > 5:
                rho, p = stats.spearmanr(pr.loc[m, "applicability_domain_distance"],
                                         pr.loc[m, "abs_error"])
                rec["spearman_ad_distance_vs_abs_error"] = rho
                rec["spearman_ad_distance_vs_abs_error_p"] = p
                rec["spearman_n"] = int(m.sum())
    ad_error_records.append(rec)

    # ---- confidence stratification ----
    pr["confidence"] = pr["predicted_proba"].apply(lambda p: max(p, 1 - p))
    pr["confidence_bin"] = pd.cut(pr["confidence"], bins=CONFIDENCE_BINS,
                                  include_lowest=True, right=False)
    for b, g in pr.groupby("confidence_bin", observed=True):
        if not len(g):
            continue
        confidence_records.append({
            "target": TARGET_LABELS[t], "target_key": t, "confidence_bin": str(b),
            "bin_lower": b.left, "n": len(g),
            "pct_of_test_set": 100.0 * len(g) / len(pr),
            "accuracy": 100.0 * g["correct"].mean(),
            "mean_confidence": g["confidence"].mean(),
        })

ad_error_df = pd.DataFrame(ad_error_records)
save_table(ad_error_df, "applicability_domain_error_analysis")
print()
print(ad_error_df.round(3).to_string(index=False))

confidence_df = pd.DataFrame(confidence_records)
save_table(confidence_df, "confidence_stratified_accuracy")
print()
print(confidence_df.pivot_table(index="target", columns="confidence_bin",
                                values="accuracy").round(1).to_string())

if len(ad_error_df):
    n_worse_outside = int((ad_error_df["error_rate_increase_outside_ad"] > 0).sum())
    print(f"\nError rate is higher outside the applicability domain for "
          f"{n_worse_outside}/{len(ad_error_df)} targets.")
if len(confidence_df):
    mono = []
    for t, g in confidence_df.groupby("target"):
        g = g.sort_values("bin_lower")
        mono.append({"target": t,
                     "accuracy_increases_monotonically": bool(g["accuracy"].is_monotonic_increasing),
                     "accuracy_lowest_bin": g["accuracy"].iloc[0],
                     "accuracy_highest_bin": g["accuracy"].iloc[-1],
                     "accuracy_gap": g["accuracy"].iloc[-1] - g["accuracy"].iloc[0]})
    confidence_monotonicity_df = pd.DataFrame(mono)
    save_table(confidence_monotonicity_df, "confidence_accuracy_monotonicity")
    print()
    print(confidence_monotonicity_df.round(2).to_string(index=False))

In [ ]:
# =============================================================================
# MODULE E (part 4) -- internal vs external degradation, all targets.
# =============================================================================
degradation_records = []
for _, r in landscape_df.iterrows():
    internal = r["internal_test_roc_auc"]
    for axis, n_col, auc_col, strength_col in (
            ("temporal (never seen in training)", "temporal_n", "temporal_roc_auc",
             "temporal_evidence_strength"),
            ("BindingDB (independent database)", "bindingdb_n", "bindingdb_roc_auc",
             "bindingdb_evidence_strength")):
        degradation_records.append({
            "target": r["target"], "target_key": r["target_key"],
            "external_axis": axis, "n_external": r[n_col],
            "internal_test_roc_auc": internal, "external_roc_auc": r[auc_col],
            "degradation_roc_auc": internal - r[auc_col] if pd.notna(r[auc_col]) else np.nan,
            "evidence_strength": r[strength_col],
        })

degradation_df = pd.DataFrame(degradation_records)
save_table(degradation_df, "internal_vs_external_degradation")
print()
print(degradation_df.round(3).to_string(index=False))

primary = degradation_df[degradation_df["evidence_strength"] == "primary"]
if len(primary):
    print(f"\nAcross rows with n >= {MIN_N_FOR_PRIMARY_CLAIM} (primary-strength evidence only), "
          f"median ROC-AUC change from internal to external: "
          f"{primary['degradation_roc_auc'].median():.3f}")
weak = degradation_df[degradation_df["evidence_strength"] == "weak (small n)"]
if len(weak):
    print(f"{len(weak)} external row(s) fall below n = {MIN_N_FOR_PRIMARY_CLAIM} and are "
          f"reported but never used as standalone claims: "
          f"{', '.join(sorted(set(weak['target'] + ' (' + weak['external_axis'].str.split(' ').str[0] + ')')))}")

## Module E2: Explainability and Reliability Bridge

Modules D and E answer neighbouring questions separately: which features the
models use, and when their predictions can be trusted. Neither says whether
those are related.

This module joins them. For each target it takes the per-compound SHAP
attributions of the deployed model, splits the held-out test set by reliability
stratum, and asks whether feature reliance differs between strata:

- high-confidence correct versus high-confidence **incorrect** predictions --
  the confidently-wrong case is the deployment-critical failure mode, and
  nothing else in the analysis characterises it
- inside versus outside the applicability domain

**Row alignment is verified, not assumed.** The stored SHAP arrays carry no row
index, so correspondence with the test predictions could only be inferred from
matching row counts. Here the first descriptor column of `shap_X_explain.npy`
is compared against the source molecular weight; a mismatch raises rather than
silently attributing the wrong compound.

In [ ]:
# =============================================================================
# MODULE E2 -- explainability/reliability bridge.
# =============================================================================
HIGH_CONFIDENCE_THRESHOLD = 0.90
RECURRENT_DESCRIPTOR_ANCHORS = ["FractionCSP3", "MW", "LogP", "HeavyAtomCount"]

bridge_records = []
for t in TARGETS:
    algo = DEPLOYED_CLF[t]
    shap_dir = results_path / f"{t}_{POOL}" / REPRESENTATION
    values_path, x_path = shap_dir / "shap_values.npy", shap_dir / "shap_X_explain.npy"
    summary_path = shap_dir / "shap_summary.json"
    if not (values_path.exists() and x_path.exists()):
        print(f"  {t}: no SHAP arrays; skipped")
        continue

    shap_algo = json.load(open(summary_path)).get("algorithm") if summary_path.exists() else None
    shap_values = np.load(values_path)
    shap_x = np.load(x_path)

    pr = preds[(preds["target"] == t) & (preds["activity_pool"] == POOL)
               & (preds["feature_representation"] == REPRESENTATION)
               & (preds["algorithm"] == algo)].reset_index(drop=True)
    if len(pr) != len(shap_values):
        print(f"  {t}: {len(shap_values)} SHAP rows vs {len(pr)} deployed predictions; skipped")
        continue

    # Row-alignment proof: descriptor column 0 must equal the source MW.
    cleaned = pd.read_csv(processed_path / f"cleaned_data_{t}_{POOL}.csv",
                          usecols=["molecule_chembl_id", "MW"])
    joined = pr.merge(cleaned, on="molecule_chembl_id", how="left")
    mw_diff = float(np.nanmax(np.abs(joined["MW"].to_numpy(float) - shap_x[:, 0])))
    if not mw_diff < 1e-6:
        raise ValueError(f"{t}: SHAP row order failed the molecular-weight alignment check "
                         f"(max difference {mw_diff}); attributions cannot be joined safely.")

    abs_shap = np.abs(shap_values)
    total = abs_shap.sum(axis=1)
    descriptor_share = np.divide(abs_shap[:, :len(DESCRIPTOR_COLS)].sum(axis=1), total,
                                 out=np.full(len(total), np.nan), where=total > 0)
    anchor_idx = [DESCRIPTOR_COLS.index(d) for d in RECURRENT_DESCRIPTOR_ANCHORS
                  if d in DESCRIPTOR_COLS]
    anchor_share = np.divide(abs_shap[:, anchor_idx].sum(axis=1), total,
                             out=np.full(len(total), np.nan), where=total > 0)

    confidence = pr["predicted_proba"].apply(lambda x: max(x, 1 - x)).to_numpy()
    correct = (pr["y_true_class"] == pr["predicted_class"]).to_numpy()
    inside = pr["within_applicability_domain"].astype(bool).to_numpy()
    hc = confidence >= HIGH_CONFIDENCE_THRESHOLD

    def _mean(values, mask):
        return float(np.nanmean(values[mask])) if mask.sum() else np.nan

    bridge_records.append({
        "target": TARGET_LABELS[t], "target_key": t,
        "deployed_algorithm": algo, "shap_artifact_algorithm": shap_algo,
        "shap_matches_deployed_model": shap_algo == algo,
        "n_test": len(pr),
        "n_high_confidence_correct": int((hc & correct).sum()),
        "n_high_confidence_incorrect": int((hc & ~correct).sum()),
        "n_inside_ad": int(inside.sum()), "n_outside_ad": int((~inside).sum()),
        "descriptor_share_hc_correct": _mean(descriptor_share, hc & correct),
        "descriptor_share_hc_incorrect": _mean(descriptor_share, hc & ~correct),
        "descriptor_share_hc_correct_minus_incorrect":
            _mean(descriptor_share, hc & correct) - _mean(descriptor_share, hc & ~correct),
        "descriptor_share_inside_ad": _mean(descriptor_share, inside),
        "descriptor_share_outside_ad": _mean(descriptor_share, ~inside),
        "descriptor_share_inside_minus_outside_ad":
            _mean(descriptor_share, inside) - _mean(descriptor_share, ~inside),
        "recurrent_anchor_share_inside_ad": _mean(anchor_share, inside),
        "recurrent_anchor_share_outside_ad": _mean(anchor_share, ~inside),
        "row_alignment_max_mw_difference": mw_diff,
    })

bridge_df = pd.DataFrame(bridge_records)
save_table(bridge_df, "explainability_reliability_bridge")

if len(bridge_df):
    show = ["target", "n_high_confidence_correct", "n_high_confidence_incorrect",
            "descriptor_share_hc_correct_minus_incorrect",
            "descriptor_share_inside_minus_outside_ad"]
    print()
    print(bridge_df[show].round(4).to_string(index=False))

    # Sign consistency matters more than magnitude at these failure-case counts.
    d = bridge_df["descriptor_share_hc_correct_minus_incorrect"].dropna()
    n_neg = int((d < 0).sum())
    print(f"\nDescriptor reliance is higher for high-confidence INCORRECT predictions in "
          f"{n_neg}/{len(d)} targets.")
    print(f"Failure-case counts are small ({bridge_df['n_high_confidence_incorrect'].min()}-"
          f"{bridge_df['n_high_confidence_incorrect'].max()} compounds); report the direction "
          f"and its consistency, not the magnitude, and always beside the counts.")
    mismatched = bridge_df[~bridge_df["shap_matches_deployed_model"]]
    if len(mismatched):
        print(f"\n  WARNING: SHAP artifact does not describe the deployed model for: "
              f"{', '.join(mismatched['target'])}. Those rows are context only.")

## Module F: Zero-Shot Cross-Target Transfer Summary

Bounded by design: a transfer matrix and one takeaway, not a re-analysis. The
controls that make the number interpretable (exact-compound overlap, scaffold
overlap, source-domain applicability status) were built into the transfer
notebook itself; this module reports the stratum that excludes source-shared
compounds when it is available, because the unfiltered stratum can be inflated by
compounds the source model has already seen.

In [ ]:
# =============================================================================
# MODULE F -- zero-shot transfer.
# =============================================================================
zs = DATA["zeroshot_headline"]

STRATUM_PREFERENCE = ["excl_source_shared", "unseen_scaffold_only", "all"]
available_strata = set(zs["stratum"].astype(str)) if "stratum" in zs.columns else set()
chosen_stratum = next((s for s in STRATUM_PREFERENCE if s in available_strata), None)
if chosen_stratum is None:
    chosen_stratum = sorted(available_strata)[0] if available_strata else None
print(f"Transfer strata available: {sorted(available_strata)}")
print(f"Reporting stratum: {chosen_stratum} "
      f"(preferring the stratum that excludes compounds shared with the source target)")

zs_sub = zs[zs["stratum"].astype(str) == chosen_stratum].copy() if chosen_stratum else zs.copy()
zs_sub["source_label"] = zs_sub["source"].map(TARGET_LABELS).fillna(zs_sub["source"])
zs_sub["destination_label"] = zs_sub["destination"].map(TARGET_LABELS).fillna(zs_sub["destination"])
save_table(zs_sub, "zeroshot_transfer_reported_stratum")

transfer_matrix = zs_sub.pivot_table(index="source_label", columns="destination_label",
                                     values="roc_auc", aggfunc="first")
order = [TARGET_LABELS[t] for t in TARGETS if TARGET_LABELS[t] in transfer_matrix.index]
col_order = [TARGET_LABELS[t] for t in TARGETS if TARGET_LABELS[t] in transfer_matrix.columns]
transfer_matrix = transfer_matrix.reindex(index=order, columns=col_order)
save_table(transfer_matrix.reset_index(), "zeroshot_transfer_matrix")
print()
print(transfer_matrix.round(3).to_string())

summary = {}
if "roc_auc_gap_vs_destination_baseline" in zs_sub.columns:
    gaps = pd.to_numeric(zs_sub["roc_auc_gap_vs_destination_baseline"], errors="coerce").dropna()
    n_beat = int((gaps > 0).sum())
    summary = {
        "stratum_reported": chosen_stratum,
        "n_directed_pairs": int(len(zs_sub)),
        "mean_transfer_roc_auc": float(pd.to_numeric(zs_sub["roc_auc"], errors="coerce").mean()),
        "mean_gap_vs_destination_baseline": float(gaps.mean()) if len(gaps) else np.nan,
        "median_gap_vs_destination_baseline": float(gaps.median()) if len(gaps) else np.nan,
        "n_pairs_beating_destination_baseline": n_beat,
        "n_pairs_evaluated_for_gap": int(len(gaps)),
        "takeaway": ("cross-target transfer is worse than the destination-specific model"
                     if len(gaps) and gaps.mean() < 0 else
                     "cross-target transfer is not consistently worse than the destination model"),
    }
    zeroshot_summary_df = pd.DataFrame([summary])
    save_table(zeroshot_summary_df, "zeroshot_transfer_takeaway")
    print()
    print(f"Directed pairs: {summary['n_directed_pairs']}  |  mean transfer ROC-AUC: "
          f"{summary['mean_transfer_roc_auc']:.3f}")
    print(f"Mean gap vs destination-specific baseline: "
          f"{summary['mean_gap_vs_destination_baseline']:.3f} "
          f"({n_beat}/{len(gaps)} pairs beat their destination baseline)")
    print(f"Takeaway: {summary['takeaway']}")
else:
    print("\nNo destination-baseline column present; transfer matrix saved without a gap summary.")

## Module G: Integrated DrugBank Screening and Docking Evidence

Closes the gap where the machine-learning validation and the structure-based
prioritization would otherwise read as two disconnected stories: one funnel from
the screened library through the ML confidence and applicability-domain gates,
into docking candidate selection, redocking-validated receptor units, consensus
poses, interaction-plausibility filtering, and the final prioritized hits.

**This module is optional by design.** If the screening or docking outputs are
absent it prints what is missing, records that in the manifest, and skips -- it
never blocks Modules B-F.

Three reporting rules are enforced here rather than left to the write-up:

1. **The headline-hit flag is reconstructed if it was not persisted.** The
   definition is consensus pass, no PAINS/Brenk structural alert, and a pass on
   the target-aware interaction check. If the upstream column exists it is used
   as-is; if not it is rebuilt from exactly those three terms so the two paths
   can never diverge.
2. **Enrichment factor at 1% is checked for resolution before it is reported.**
   With roughly 100 compounds per docking unit the top 1% is a single molecule,
   so the metric can take only two values and carries almost no information. The
   cell tests this on the actual data and excludes the metric when it holds.
3. **Enrichment metrics travel with the property-bias audit.** Docking scores
   track molecular size and lipophilicity, so an enrichment result computed on
   actives and inactives that differ in those properties is partly a property
   effect. The largest standardized mean difference per target is joined onto
   the enrichment summary so the two are never quoted apart.

In [ ]:
# =============================================================================
# MODULE G -- availability gate.
# =============================================================================
G_REQUIRED_ANY = ["screening_novel_summary", "docking_final", "docking_candidate_sel"]
g_available = {k: (k in DATA) for k, _, sev, mod in INPUTS if mod == "G"}
MODULE_G_ENABLED = any(g_available.get(k, False) for k in G_REQUIRED_ANY)

print("Module G input availability:")
for k, v in sorted(g_available.items()):
    print(f"  {'present' if v else 'MISSING'}: {k}")

if not MODULE_G_ENABLED:
    print("\nModule G skipped: neither the DrugBank screening summary nor the docking "
          "results are available. Modules B-F above are unaffected and complete.")
else:
    print("\nModule G enabled.")

In [ ]:
# =============================================================================
# MODULE G (part 1) -- headline-hit reconstruction and docking outcome tables.
# =============================================================================
docking_final_df = None
headline_hits_df = None
HEADLINE_DEFINITION = ("consensus_pass AND NOT has_pains_or_brenk_alert AND passes_plif_check")

if MODULE_G_ENABLED and "docking_final" in DATA:
    docking_final_df = DATA["docking_final"].copy()

    # Notebook 5 now persists passes_plif_check, but older in-memory/checkpointed
    # outputs may still carry the raw Module J column name. Normalize once here
    # so the headline definition cannot silently diverge by run vintage.
    if ("passes_plif_check" not in docking_final_df.columns
            and "passes_target_aware_plif_check" in docking_final_df.columns):
        docking_final_df["passes_plif_check"] = docking_final_df["passes_target_aware_plif_check"]

    if "is_headline_hit" in docking_final_df.columns:
        headline_source = "persisted upstream column"
        docking_final_df["is_headline_hit"] = docking_final_df["is_headline_hit"].astype(bool)
    else:
        needed = ["consensus_pass", "has_pains_or_brenk_alert", "passes_plif_check"]
        missing = [c for c in needed if c not in docking_final_df.columns]
        if missing:
            raise ValueError(
                "Cannot reconstruct docking headline hits because docking_results_final.csv "
                f"is missing required column(s): {missing}. Rerun notebook 5 Module K "
                "after Module J so the consensus, structural-alert and PLIF gates are all persisted."
            )
        headline_source = "reconstructed in this notebook from the three-term definition"
        docking_final_df["is_headline_hit"] = (
            docking_final_df["consensus_pass"].fillna(False).astype(bool)
            & (docking_final_df["has_pains_or_brenk_alert"] == False)
            & (docking_final_df["passes_plif_check"] == True)
        )
    if "headline_hit_definition" not in docking_final_df.columns:
        docking_final_df["headline_hit_definition"] = HEADLINE_DEFINITION

    print(f"Headline-hit flag: {headline_source}")
    print(f"Definition: {HEADLINE_DEFINITION}")
    print(f"  rows: {len(docking_final_df)}")
    print(f"  consensus pass: {int(docking_final_df['consensus_pass'].fillna(False).astype(bool).sum())}")
    if "passes_plif_check" in docking_final_df.columns:
        print(f"  passes target-aware interaction check: "
              f"{int((docking_final_df['passes_plif_check'] == True).sum())}")
    print(f"  headline hits: {int(docking_final_df['is_headline_hit'].sum())}")

    headline_hits_df = docking_final_df[docking_final_df["is_headline_hit"]].copy()
    keep_cols = [c for c in ["unit", "target", "state", "global_compound_id", "drugbank_id",
                             "generic_name", "vina_affinity", "gnina_cnn_score",
                             "priority_score_from_notebook4", "is_unanimous_consensus_notebook4",
                             "n_protein_contacts", "n_polar_contacts", "n_ionic_acid_contacts",
                             "target_plif_rule", "lipinski_pass", "muegge_pass",
                             "receptor_prep_provisional", "fusion_boundary_confirmed",
                             "fusion_excision_range", "headline_hit_definition"]
                 if c in headline_hits_df.columns]
    headline_hits_df = headline_hits_df[keep_cols]
    save_table(headline_hits_df, "docking_headline_hits")
    save_table(docking_final_df, "docking_results_final_with_headline_flag")

    if "receptor_prep_provisional" in headline_hits_df.columns:
        n_prov = int((headline_hits_df["receptor_prep_provisional"] == True).sum())
        if n_prov:
            print(f"\n  {n_prov} headline hit row(s) sit on receptor units whose fusion-excision "
                  f"boundary is unconfirmed; they carry a provisional flag into every table below "
                  f"and must be disclosed as provisional in the manuscript.")
    if len(headline_hits_df):
        print()
        print(headline_hits_df.head(15).to_string(index=False))

In [ ]:
# =============================================================================
# MODULE G (part 2) -- the screening-to-docking funnel, one row per target.
# =============================================================================
funnel_df = None
if MODULE_G_ENABLED:
    rows = []
    novel = DATA.get("screening_novel_summary")
    screening_candidates = DATA.get("screening_candidates")
    cand_sel = DATA.get("docking_candidate_sel")
    redock = DATA.get("docking_redocking")

    # The ranked DrugBank candidate file is the authoritative source for the
    # screening-stage counts. If an older summary file is missing or stale, rebuild
    # it here so the funnel cannot mix model vintages.
    derived_novel = None
    required_screen_cols = {"target", "drugbank_id", "within_applicability_domain",
                            "predicted_proba_calibrated", "activity_pool",
                            "feature_representation", "algorithm"}
    if screening_candidates is not None and required_screen_cols.issubset(screening_candidates.columns):
        # Verify that the funnel uses exactly the corrected Notebook 4 production artifact.
        if not screening_candidates["activity_pool"].eq("full").all():
            raise ValueError("Screening funnel input contains non-FULL activity-pool rows.")
        if not screening_candidates["feature_representation"].eq("combined").all():
            raise ValueError("Screening funnel input contains non-COMBINED representation rows.")
        for _t, _grp in screening_candidates.groupby("target"):
            _algos = sorted(_grp["algorithm"].dropna().astype(str).str.strip().unique())
            if len(_algos) != 1 or _algos[0] != str(DEPLOYED_CLF.get(_t)):
                raise ValueError(
                    f"Screening funnel provenance mismatch for {_t}: observed {_algos}, "
                    f"expected {DEPLOYED_CLF.get(_t)}."
                )

        derived_novel = (
            screening_candidates.groupby("target", as_index=False)
            .agg(n_rows=("drugbank_id", "size"),
                 n_unique_drugs=("drugbank_id", "nunique"),
                 pct_within_ad=("within_applicability_domain", "mean"),
                 median_proba=("predicted_proba_calibrated", "median"))
        )
        derived_novel = derived_novel.sort_values("target").reset_index(drop=True)

        summary_path = screen_path / "analysis_summary_novel_by_target.csv"
        replace_summary = novel is None
        if novel is not None and {"target", "n_rows"}.issubset(novel.columns):
            lhs = novel[["target", "n_rows"]].sort_values("target").reset_index(drop=True)
            rhs = derived_novel[["target", "n_rows"]].sort_values("target").reset_index(drop=True)
            replace_summary = not lhs.equals(rhs)
        if replace_summary:
            derived_novel.to_csv(summary_path, index=False)
            _register("analysis_summary_novel_by_target.csv", summary_path, len(derived_novel))
            DATA["screening_novel_summary"] = derived_novel
            novel = derived_novel
            print("  rebuilt screening novel summary from drugbank_primary_ranked_candidates.csv")
        else:
            novel = derived_novel
            DATA["screening_novel_summary"] = derived_novel
            print("  screening novel summary verified against drugbank_primary_ranked_candidates.csv")

    for t in TARGETS:
        rec = {"target": TARGET_LABELS[t], "target_key": t,
               "deployed_algorithm": DEPLOYED_CLF[t]}

        if novel is not None and "target" in novel.columns:
            n_row = novel[novel["target"] == t]
            if len(n_row):
                r = n_row.iloc[0]
                rec["n_high_confidence_novel_rows"] = r.get("n_rows", np.nan)
                rec["n_high_confidence_novel_unique_drugs"] = r.get("n_unique_drugs", np.nan)
                rec["pct_novel_within_applicability_domain"] = r.get("pct_within_ad", np.nan)
                rec["median_calibrated_probability"] = r.get("median_proba", np.nan)

        if cand_sel is not None and "target" in cand_sel.columns:
            c_row = cand_sel[cand_sel["target"] == t]
            if len(c_row):
                r = c_row.iloc[0]
                rec["n_after_docking_hard_gates"] = r.get("available_rows_after_hard_gates", np.nan)
                rec["n_unique_after_deduplication"] = r.get("unique_compounds_after_deduplication", np.nan)
                rec["n_selected_for_docking"] = r.get("selected_for_docking", np.nan)
                rec["docking_candidate_cap"] = r.get("target_candidate_cap", np.nan)
                rec["underfilled_vs_cap"] = r.get("underfilled_target", np.nan)

        if redock is not None and "target" in redock.columns:
            r_sub = redock[redock["target"] == t]
            rec["n_receptor_units"] = len(r_sub)
            if len(r_sub) and "unit_passed" in r_sub.columns:
                rec["n_receptor_units_redocking_passed"] = int(r_sub["unit_passed"].astype(bool).sum())
                rec["max_redocking_rmsd_angstrom"] = r_sub["best_rmsd"].max()
                rec["max_redocking_attempts_used"] = (r_sub["n_attempts_used"].max()
                                                      if "n_attempts_used" in r_sub.columns else np.nan)

        if docking_final_df is not None and "target" in docking_final_df.columns:
            d_sub = docking_final_df[docking_final_df["target"] == t]
            rec["n_docked_unit_candidate_pairs"] = len(d_sub)
            if len(d_sub):
                rec["n_consensus_pass"] = int(d_sub["consensus_pass"].fillna(False).astype(bool).sum())
                if "passes_plif_check" in d_sub.columns:
                    rec["n_passes_interaction_check"] = int((d_sub["passes_plif_check"] == True).sum())
                rec["n_headline_hits"] = int(d_sub["is_headline_hit"].sum())
                rec["n_unique_headline_compounds"] = int(
                    d_sub.loc[d_sub["is_headline_hit"], "global_compound_id"].nunique())
            else:
                rec["n_consensus_pass"] = 0
                rec["n_passes_interaction_check"] = 0
                rec["n_headline_hits"] = 0
                rec["n_unique_headline_compounds"] = 0
        rows.append(rec)

    funnel_df = pd.DataFrame(rows)
    save_table(funnel_df, "screening_docking_funnel")
    print()
    print(funnel_df.to_string(index=False))

    zero_candidate_targets = []
    if "n_selected_for_docking" in funnel_df.columns:
        zero_candidate_targets = funnel_df.loc[
            pd.to_numeric(funnel_df["n_selected_for_docking"], errors="coerce").fillna(0) == 0,
            "target"].tolist()
    if zero_candidate_targets:
        print(f"\n  {', '.join(zero_candidate_targets)} contributed no docking candidate. This is "
              f"an upstream screening outcome, not a docking failure: no library compound passed "
              f"the combined high-confidence, in-applicability-domain and not-already-in-training "
              f"gates. Report it as a result of the funnel, and note that the affected target still "
              f"participates in the docking validation arm below.")


In [ ]:
# =============================================================================
# MODULE G (part 3) -- docking validation: discrimination, property bias,
# ML agreement, rediscovery.
# =============================================================================
discriminative_summary_df = None
if MODULE_G_ENABLED and "docking_discriminative" in DATA:
    disc = DATA["docking_discriminative"].copy()

    # ---- Resolution check on enrichment factor at 1% ----
    ef_notes = []
    for col, frac in (("ef_1pct", 0.01), ("ef_5pct", 0.05), ("ef_10pct", 0.10)):
        if col not in disc.columns:
            continue
        vals = pd.to_numeric(disc[col], errors="coerce").dropna()
        n_typical = pd.to_numeric(disc.get("n_compounds"), errors="coerce").median()
        n_in_top = float(np.floor(n_typical * frac)) if pd.notna(n_typical) else np.nan
        usable = bool(vals.nunique() > 2 and (n_in_top >= 3))
        ef_notes.append({
            "metric": col, "top_fraction": frac,
            "median_n_compounds_per_unit": n_typical,
            "compounds_in_top_fraction": n_in_top,
            "n_distinct_values_observed": int(vals.nunique()),
            "usable_at_this_sample_size": usable,
            "reason": ("" if usable else
                       "the top fraction contains too few compounds for the metric to take more "
                       "than a couple of distinct values; it carries almost no information at this n"),
        })
    ef_resolution_df = pd.DataFrame(ef_notes)
    save_table(ef_resolution_df, "enrichment_metric_resolution_check")
    print("Enrichment-metric resolution check:")
    print(ef_resolution_df.to_string(index=False))
    UNUSABLE_EF = set(ef_resolution_df.loc[~ef_resolution_df["usable_at_this_sample_size"], "metric"])

    disc["auc_ci_excludes_chance"] = (pd.to_numeric(disc.get("auc_ci_low"), errors="coerce") > 0.5)
    disc["target_key"] = disc["unit"].astype(str).str.split("_").str[0]

    # ---- join the largest property imbalance per target ----
    bias = DATA.get("docking_property_bias")
    if bias is not None and "abs_standardized_mean_difference" in bias.columns:
        worst_bias = (bias.sort_values("abs_standardized_mean_difference", ascending=False)
                      .groupby("target", as_index=False).first()
                      [["target", "property", "standardized_mean_difference",
                        "abs_standardized_mean_difference"]]
                      .rename(columns={"target": "target_key",
                                       "property": "largest_imbalance_property",
                                       "standardized_mean_difference": "largest_imbalance_smd",
                                       "abs_standardized_mean_difference": "largest_imbalance_abs_smd"}))
        disc = disc.merge(worst_bias, on="target_key", how="left")
        disc["enrichment_confounded_by_property_imbalance"] = (
            pd.to_numeric(disc.get("largest_imbalance_abs_smd"), errors="coerce") >= 0.5)

    drop_cols = [c for c in disc.columns if c in UNUSABLE_EF]
    discriminative_summary_df = disc.drop(columns=drop_cols)
    if drop_cols:
        print(f"\nExcluded from the reported enrichment summary: {', '.join(drop_cols)} "
              f"(see the resolution check above). The values remain in the docking notebook's "
              f"own output; they are withheld here rather than reported without resolution.")
    save_table(discriminative_summary_df, "docking_discriminative_validation_summary")

    print()
    show = [c for c in ["unit", "score_type", "auc", "auc_ci_low", "auc_ci_high",
                        "auc_ci_excludes_chance", "largest_imbalance_property",
                        "largest_imbalance_abs_smd"] if c in discriminative_summary_df.columns]
    print(discriminative_summary_df[show].round(3).to_string(index=False))

    by_engine = (discriminative_summary_df.groupby("score_type")
                 .agg(n_units=("auc", "size"), mean_auc=("auc", "mean"),
                      min_auc=("auc", "min"), max_auc=("auc", "max"),
                      n_units_ci_excludes_chance=("auc_ci_excludes_chance", "sum"))
                 .reset_index())
    save_table(by_engine, "docking_discrimination_by_engine")
    print()
    print(by_engine.round(3).to_string(index=False))
    print("\nRead this as ranking power, not pose quality: a unit whose confidence interval "
          "includes 0.5 provides no evidence that its docking score separates actives from "
          "inactives, however well the same unit reproduced its crystal pose.")

In [ ]:
# =============================================================================
# MODULE G (part 4) -- ML/docking agreement and production-path rediscovery.
# =============================================================================
if MODULE_G_ENABLED and "docking_ml_correlation" in DATA:
    mlc = DATA["docking_ml_correlation"].copy()
    mlc["target_key"] = mlc["unit"].astype(str).str.split("_").str[0]

    # The correlation file must stamp its own ML algorithm provenance. Do not
    # infer it from a sibling enrichment-sampling file: cached/rerun outputs can
    # easily have different timestamps and therefore different provenance.
    if "algorithm_used" in mlc.columns:
        mlc["algorithm_used"] = mlc["algorithm_used"].fillna("").astype(str)
        mlc["expected_deployed_algorithm"] = mlc["target_key"].map(DEPLOYED_CLF)
        mlc["ml_side_used_deployed_model"] = (
            mlc["algorithm_used"] == mlc["expected_deployed_algorithm"].astype(str)
        )
        mlc["interpretation_caveat"] = np.where(
            mlc["ml_side_used_deployed_model"], "",
            "ML probabilities for this target came from an algorithm other than the deployed one; "
            "recompute before quoting this correlation")
    else:
        mlc["algorithm_used"] = "UNSTAMPED"
        mlc["expected_deployed_algorithm"] = mlc["target_key"].map(DEPLOYED_CLF)
        mlc["ml_side_used_deployed_model"] = False
        mlc["interpretation_caveat"] = (
            "Correlation file does not stamp algorithm_used; provenance cannot be verified from "
            "this CSV. Rerun notebook 5's ML-vs-docking correlation cell after the deployed-"
            "algorithm fix before citing these rows."
        )
    save_table(mlc, "ml_docking_agreement_summary")
    print(mlc[["unit", "score_type", "n_compounds", "algorithm_used", "expected_deployed_algorithm",
               "spearman_rho", "spearman_p", "ml_side_used_deployed_model"]].round(4).to_string(index=False))
    n_flagged = int((~mlc["ml_side_used_deployed_model"]).sum())
    if n_flagged:
        print(f"\n  {n_flagged} correlation row(s) are flagged: the machine-learning side was "
              f"computed with an algorithm other than the deployed one for that target. The "
              f"label-based enrichment metrics above are unaffected.")

if MODULE_G_ENABLED and "docking_rediscovery" in DATA:
    redisc = DATA["docking_rediscovery"].copy()
    save_table(redisc, "docking_rediscovery_summary")
    print()
    show = [c for c in ["unit", "drugbank_id", "vina_rmsd_vs_crystal_reference",
                        "gnina_rmsd_vs_crystal_reference", "vina_rank_percentile_vs_candidates",
                        "gnina_rank_percentile_vs_candidates", "overall_rediscovery_pass"]
            if c in redisc.columns]
    print(redisc[show].round(3).to_string(index=False))
    print("\nRank percentile is the fraction of production candidates scoring at least as well, "
          "so 0.00 means better than every candidate and 1.00 means worse than all of them. "
          "Where a reference ligand reproduces its crystal pose while ranking poorly, pose "
          "accuracy and score ranking are decoupled for that unit, which is evidence about what "
          "the docking layer can and cannot be asked to do.")

## Module H: Figures, Tables and Manifest

Produces the paper and SI candidate outputs. It deliberately does **not** decide
what goes in the main text and what goes to supplementary: the project convention
is to over-generate here and triage at write-up, when the argument of the paper is
settled and the display budget is known.

Every figure is written as 300 dpi PNG plus vector PDF, and the dataframe behind
each one is written beside it as `<stem>_source.csv`, so no panel can end up in
the state where the image exists but the numbers that produced it do not.

In [ ]:
# =============================================================================
# FIGURE 1 -- cross-target performance: internal vs both external axes.
# =============================================================================
fig_df = landscape_df[["target", "internal_test_roc_auc", "temporal_roc_auc", "temporal_n",
                       "bindingdb_roc_auc", "bindingdb_n", "bindingdb_evidence_strength",
                       "temporal_evidence_strength", "internal_test_r2"]].copy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
x = np.arange(len(fig_df))
w = 0.27
series = [("internal_test_roc_auc", "Internal scaffold-split test", BLUE),
          ("temporal_roc_auc", "Temporal holdout, never seen", ORANGE),
          ("bindingdb_roc_auc", "Independent database", GREEN)]
for i, (col, label, color) in enumerate(series):
    ax.bar(x + (i - 1) * w, pd.to_numeric(fig_df[col], errors="coerce"),
           width=w, label=label, color=color, edgecolor="white", linewidth=0.5)
# Annotate the sample size behind each external bar so a 6-compound result can
# never be read at the same weight as a 300-compound one.
for i, (col, n_col) in enumerate([("temporal_roc_auc", "temporal_n"),
                                  ("bindingdb_roc_auc", "bindingdb_n")], start=1):
    for xi, (v, n) in enumerate(zip(pd.to_numeric(fig_df[col], errors="coerce"),
                                    pd.to_numeric(fig_df[n_col], errors="coerce"))):
        if pd.notna(v) and pd.notna(n):
            ax.text(xi + (i - 1) * w, v + 0.012, f"n={int(n)}", ha="center",
                    va="bottom", fontsize=6.5, rotation=90)
ax.axhline(0.5, color=GREY, linestyle="--", linewidth=0.8, zorder=0)
ax.set_xticks(x); ax.set_xticklabels(fig_df["target"])
ax.set_ylabel("Area under the ROC curve")
ax.set_ylim(0.4, 1.06)
ax.legend(frameon=False, loc="lower right")
panel_label(ax, "A")

ax = axes[1]
ax.bar(x, pd.to_numeric(fig_df["internal_test_r2"], errors="coerce"),
       width=0.55, color=BLUE, edgecolor="white", linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(fig_df["target"])
ax.set_ylabel("Coefficient of determination, held-out test")
ax.set_ylim(0, 1.0)
panel_label(ax, "B")

fig.tight_layout()
save_fig(fig, "fig7_cross_target_performance", fig_df)

In [ ]:
# =============================================================================
# FIGURE 2 -- dataset properties vs performance (core question 2).
# =============================================================================
scatter_specs = [
    ("final_unique_compounds", "Curated compounds", True),
    ("pct_singleton_scaffolds", "Singleton scaffolds (%)", False),
    ("active_pct", "Active compounds (%)", False),
    ("assay_support_pct_single", "Compounds with single-assay support (%)", False),
]
scatter_specs = [(c, lab, log) for c, lab, log in scatter_specs
                 if c in target_properties_df.columns
                 and pd.to_numeric(target_properties_df[c], errors="coerce").notna().sum() >= 3]

if scatter_specs:
    ncol = min(2, len(scatter_specs))
    nrow = int(np.ceil(len(scatter_specs) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 4.0 * nrow), squeeze=False)
    for i, (col, label, use_log) in enumerate(scatter_specs):
        ax = axes[i // ncol][i % ncol]
        xv = pd.to_numeric(target_properties_df[col], errors="coerce")
        yv = pd.to_numeric(target_properties_df["internal_test_roc_auc"], errors="coerce")
        ax.scatter(xv, yv, s=70, color=BLUE, zorder=3)
        for _, r in target_properties_df.iterrows():
            if pd.notna(r[col]) and pd.notna(r["internal_test_roc_auc"]):
                ax.annotate(r["target_label"], (r[col], r["internal_test_roc_auc"]),
                            textcoords="offset points", xytext=(6, 4), fontsize=8)
        m = xv.notna() & yv.notna()
        if m.sum() >= 3:
            rho, _ = stats.spearmanr(xv[m], yv[m])
            ax.text(0.03, 0.06, f"Spearman rho = {rho:.2f} (n = {int(m.sum())})",
                    transform=ax.transAxes, fontsize=8, color="#444444")
        if use_log:
            ax.set_xscale("log")
        ax.set_xlabel(label)
        ax.set_ylabel("Test area under the ROC curve")
        panel_label(ax, "ABCD"[i])
    for j in range(len(scatter_specs), nrow * ncol):
        axes[j // ncol][j % ncol].axis("off")
    fig.tight_layout()
    save_fig(fig, "fig7_dataset_property_correlates", target_properties_df)
else:
    print("No dataset property columns with enough coverage to plot; figure 2 skipped.")

In [ ]:
# =============================================================================
# FIGURE 3 -- reliability and uncertainty across targets (core question 4).
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
x = np.arange(len(calibration_cross_target_df))

ax = axes[0][0]
ax.bar(x - 0.2, calibration_cross_target_df["ece_raw"], width=0.4,
       label="Before calibration", color=GREY, edgecolor="white", linewidth=0.5)
ax.bar(x + 0.2, calibration_cross_target_df["ece_calibrated"], width=0.4,
       label="After calibration", color=BLUE, edgecolor="white", linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(calibration_cross_target_df["target"])
ax.set_ylabel("Expected calibration error")
ax.legend(frameon=False)
panel_label(ax, "A")

ax = axes[0][1]
clf_conf = conformal_cross_target_df[
    (conformal_cross_target_df["task"] == "classification")
    & (conformal_cross_target_df["nominal_level"] == CONFIDENCE_LEVEL)
    & (conformal_cross_target_df["stratum"].isin(["overall", "within_ad", "outside_ad"]))]
piv = clf_conf.pivot_table(index="target", columns="stratum", values="empirical_coverage")
piv = piv.reindex([TARGET_LABELS[t] for t in TARGETS if TARGET_LABELS[t] in piv.index])
stratum_labels = {"overall": "All test compounds", "within_ad": "Inside applicability domain",
                  "outside_ad": "Outside applicability domain"}
xs = np.arange(len(piv))
for i, s in enumerate([c for c in ["overall", "within_ad", "outside_ad"] if c in piv.columns]):
    ax.bar(xs + (i - 1) * 0.27, piv[s], width=0.27, label=stratum_labels[s],
           color=[BLUE, GREEN, ORANGE][i], edgecolor="white", linewidth=0.5)
ax.axhline(CONFIDENCE_LEVEL, color=RED, linestyle="--", linewidth=1.0,
           label=f"Nominal level ({CONFIDENCE_LEVEL:.2f})")
ax.set_xticks(xs); ax.set_xticklabels(piv.index)
ax.set_ylabel("Empirical coverage")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=7.5)
panel_label(ax, "B")

ax = axes[1][0]
for i, (t, g) in enumerate(confidence_df.groupby("target")):
    g = g.sort_values("bin_lower")
    ax.plot(g["bin_lower"], g["accuracy"], marker="o", markersize=4,
            color=PALETTE[i % len(PALETTE)], label=t, linewidth=1.4)
ax.set_xlabel("Prediction confidence, lower bin edge")
ax.set_ylabel("Accuracy (%)")
ax.legend(frameon=False, fontsize=8)
panel_label(ax, "C")

ax = axes[1][1]
xs = np.arange(len(ad_error_df))
ax.bar(xs - 0.2, ad_error_df["error_rate_within_ad"], width=0.4,
       label="Inside applicability domain", color=BLUE, edgecolor="white", linewidth=0.5)
ax.bar(xs + 0.2, ad_error_df["error_rate_outside_ad"], width=0.4,
       label="Outside applicability domain", color=ORANGE, edgecolor="white", linewidth=0.5)
for xi, n_out in enumerate(ad_error_df["n_outside_ad"]):
    ax.text(xi + 0.2, 0.4, f"n={int(n_out)}", ha="center", va="bottom", fontsize=6.5, rotation=90)
ax.set_xticks(xs); ax.set_xticklabels(ad_error_df["target"])
ax.set_ylabel("Misclassification rate (%)")
ax.legend(frameon=False, fontsize=8)
panel_label(ax, "D")

fig.tight_layout()
reliability_source = calibration_cross_target_df.merge(
    ad_error_df.drop(columns=[c for c in ["target_key"] if c in ad_error_df.columns]),
    on="target", how="outer")
save_fig(fig, "fig7_reliability_uncertainty", reliability_source)

In [ ]:
# =============================================================================
# FIGURE 4 -- SHAP descriptor importance heatmap across targets (question 3).
# FIGURE 5 -- feature representation comparison.
# =============================================================================
heat = globals().get("shap_heatmap_combined")
if heat is not None and heat.size:
    fig, ax = plt.subplots(figsize=(1.35 * len(heat.columns) + 3.0, 0.45 * len(heat.index) + 2.2))
    data = heat.astype(float).values
    im = ax.imshow(data, cmap="Blues", aspect="auto")
    ax.set_xticks(np.arange(len(heat.columns))); ax.set_xticklabels(heat.columns)
    ax.set_yticks(np.arange(len(heat.index))); ax.set_yticklabels(heat.index)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                        color="white" if v > np.nanmax(data) * 0.6 else "#222222")
    cb = fig.colorbar(im, ax=ax, shrink=0.85)
    cb.set_label("Share of top-feature importance")
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.tight_layout()
    save_fig(fig, "fig7_shap_descriptor_heatmap", heat.reset_index())
else:
    print("No descriptor-level SHAP matrix available; figure 4 skipped.")

rep_plot = representation_df.dropna(subset=["test_roc_auc_deployed_algorithm"])
if len(rep_plot):
    fig, ax = plt.subplots(figsize=(8.2, 4.2))
    targets_order = [TARGET_LABELS[t] for t in TARGETS if TARGET_LABELS[t] in set(rep_plot["target"])]
    xs = np.arange(len(targets_order))
    for i, rep in enumerate(["morgan", "descriptors", "combined"]):
        sub = (rep_plot[rep_plot["feature_representation"] == rep]
               .set_index("target").reindex(targets_order))
        ax.bar(xs + (i - 1) * 0.27, sub["test_roc_auc_deployed_algorithm"], width=0.27,
               label=FEATURE_REP_LABELS[rep], color=[GREY, ORANGE, BLUE][i],
               edgecolor="white", linewidth=0.5)
    ax.set_xticks(xs); ax.set_xticklabels(targets_order)
    ax.set_ylabel("Test area under the ROC curve")
    ax.set_ylim(0.5, 1.0)
    ax.legend(frameon=False)
    fig.tight_layout()
    save_fig(fig, "fig7_feature_representation_comparison", rep_plot)

In [ ]:
# =============================================================================
# FIGURE 6 -- zero-shot transfer matrix.
# =============================================================================
if transfer_matrix is not None and transfer_matrix.size:
    fig, ax = plt.subplots(figsize=(1.15 * len(transfer_matrix.columns) + 3.2,
                                    0.85 * len(transfer_matrix.index) + 2.6))
    data = transfer_matrix.astype(float).values
    im = ax.imshow(data, cmap="RdYlBu", vmin=0.4, vmax=1.0, aspect="auto")
    ax.set_xticks(np.arange(len(transfer_matrix.columns)))
    ax.set_xticklabels(transfer_matrix.columns)
    ax.set_yticks(np.arange(len(transfer_matrix.index)))
    ax.set_yticklabels(transfer_matrix.index)
    ax.set_xlabel("Evaluated on")
    ax.set_ylabel("Trained on")
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="#222222")
    cb = fig.colorbar(im, ax=ax, shrink=0.85)
    cb.set_label("Area under the ROC curve")
    fig.tight_layout()
    save_fig(fig, "fig7_zeroshot_transfer_matrix", transfer_matrix.reset_index())
else:
    print("No transfer matrix available; figure 6 skipped.")

In [ ]:
# =============================================================================
# FIGURE 7 -- screening-to-docking funnel (Module G only).
# =============================================================================
if MODULE_G_ENABLED and funnel_df is not None and len(funnel_df):
    stages = [("n_high_confidence_novel_unique_drugs", "High-confidence novel drugs"),
              ("n_after_docking_hard_gates", "Passing docking entry gates"),
              ("n_selected_for_docking", "Selected for docking"),
              ("n_consensus_pass", "Consensus pose across engines"),
              ("n_passes_interaction_check", "Passing interaction check"),
              ("n_headline_hits", "Prioritized hits")]
    stages = [(c, lab) for c, lab in stages if c in funnel_df.columns]
    if stages:
        fig, ax = plt.subplots(figsize=(9.0, 4.6))
        xs = np.arange(len(stages))
        for i, t in enumerate(funnel_df["target"]):
            vals = [pd.to_numeric(funnel_df.loc[funnel_df["target"] == t, c],
                                  errors="coerce").iloc[0] for c, _ in stages]
            ax.plot(xs, vals, marker="o", markersize=5, linewidth=1.5,
                    color=PALETTE[i % len(PALETTE)], label=t)
        ax.set_xticks(xs)
        ax.set_xticklabels([lab for _, lab in stages], rotation=20, ha="right")
        ax.set_yscale("symlog", linthresh=1)
        ax.set_ylabel("Compounds remaining")
        ax.legend(frameon=False, fontsize=8)
        fig.tight_layout()
        save_fig(fig, "fig7_screening_docking_funnel", funnel_df)
else:
    print("Module G not enabled; funnel figure skipped.")

In [ ]:
# =============================================================================
# PAPER/SI CANDIDATE TABLES -- written to the shared outputs directory.
# Main-text versus supplementary triage happens at write-up, not here.
# =============================================================================
table_specs = [
    ("table_dataset_characteristics", target_properties_df),
    ("table_model_performance_landscape", landscape_df),
    ("table_cross_target_consistency", consistency_df),
    ("table_performance_drivers", driver_corr_df.head(25)),
    ("table_feature_representation", representation_wide_df),
    ("table_calibration_reliability", calibration_cross_target_df),
    ("table_conformal_coverage", conformal_cross_target_df),
    ("table_applicability_domain_error", ad_error_df),
    ("table_internal_vs_external", degradation_df),
    ("table_zeroshot_transfer_matrix", transfer_matrix.reset_index()
     if transfer_matrix is not None and transfer_matrix.size else pd.DataFrame()),
]
if globals().get("feature_transferability_df") is not None:
    table_specs.append(("table_feature_transferability",
                        feature_transferability_df.head(40)))
if MODULE_G_ENABLED:
    if funnel_df is not None:
        table_specs.append(("table_screening_docking_funnel", funnel_df))
    if headline_hits_df is not None:
        table_specs.append(("table_docking_headline_hits", headline_hits_df))
    if discriminative_summary_df is not None:
        table_specs.append(("table_docking_discrimination", discriminative_summary_df))

for stem, df in table_specs:
    if df is None or not len(df):
        print(f"  skipped (empty): {stem}")
        continue
    save_table(df, stem, out_dir=out_tab_dir)

In [ ]:
# =============================================================================
# MANIFEST -- SHA-256 over every artifact this notebook wrote.
# =============================================================================
manifest_outputs = {}
for name, entry in all_outputs.items():
    p = Path(entry["path"])
    if not p.exists():
        continue
    rec = {"path": str(p), "sha256": _sha256_file(p), "bytes": p.stat().st_size}
    if "n_rows" in entry:
        rec["n_rows"] = entry["n_rows"]
    manifest_outputs[name] = rec

manifest = {
    "notebook": "07_meta_analysis_patched_provenance.ipynb",
    "timestamp": datetime.datetime.now().isoformat(),
    "python_version": platform.python_version(),
    "targets": TARGETS,
    "headline_pool": POOL,
    "headline_representation": REPRESENTATION,
    "conformal_confidence_level": CONFIDENCE_LEVEL,
    "min_n_for_primary_external_claim": MIN_N_FOR_PRIMARY_CLAIM,
    "deployed_algorithm_source": (
        "classification: Notebook 4 drugbank_primary_ranked_candidates.csv FULL/combined provenance "
        "when available, otherwise Notebook 3 best_algorithm_by_combination.csv fallback; "
        "regression: Notebook 3 best_algorithm_by_combination.csv"
    ),
    "deployed_algorithm_classification_source_by_target": DEPLOYED_CLF_SOURCE,
    "notebook4_primary_screen_provenance_available": bool(PRIMARY_SCREEN_PROVENANCE_AVAILABLE),
    "selection_table_sha256": SELECTION_TABLE_SHA256,
    "selection_table_reproducible_from_tuning_history": bool(
        selection_check_df["agree"].all()) if len(selection_check_df) else None,
    "explainability_reliability_bridge_targets": (
        bridge_df["target"].tolist() if "bridge_df" in globals() and len(bridge_df) else []),
    "deployed_algorithm_classification": DEPLOYED_CLF,
    "deployed_algorithm_regression": DEPLOYED_REG,
    "deployed_algorithm_disagreements_detected": sorted(ALGO_DISAGREEMENT_TARGETS),
    "module_g_enabled": bool(MODULE_G_ENABLED),
    "headline_hit_definition": HEADLINE_DEFINITION,
    "headline_hit_flag_source": (globals().get("headline_source", "n/a")
                                 if MODULE_G_ENABLED else "n/a"),
    "optional_inputs_unavailable": [k for k, _, note in missing_optional],
    "n_outputs": len(manifest_outputs),
    "outputs": manifest_outputs,
}
manifest_path = meta_path / "manifest_07_meta_analysis.json"
with open(manifest_path, "w") as fh:
    json.dump(manifest, fh, indent=2, default=str)
print(f"Manifest saved: {manifest_path}")
print(f"Artifacts recorded: {len(manifest_outputs)}")

In [ ]:
# =============================================================================
# CLOSING SUMMARY -- what this notebook established, in the order the paper
# will need it.
# =============================================================================
print("Notebook 7 (cross-target meta-analysis) complete.\n")

auc = pd.to_numeric(landscape_df["internal_test_roc_auc"], errors="coerce")
print(f"Question 1 (consistency across targets): internal test ROC-AUC spans "
      f"{auc.min():.3f}-{auc.max():.3f} across {len(landscape_df)} targets "
      f"(range {auc.max() - auc.min():.3f}).")

if len(driver_corr_df):
    top = driver_corr_df.iloc[0]
    n_tied = int(top.get("n_properties_tied_at_this_rho", 1))
    tie_note = (f" (tied with {n_tied - 1} other property/metric pair(s) at the same rho)"
                if n_tied > 1 else "")
    print(f"Question 2 (what drives performance): strongest property association is "
          f"rho = {top['spearman_rho']:.2f}{tie_note}, led by {top['property']} vs "
          f"{top['performance_metric']} at n = {int(top['n_targets'])} targets -- descriptive, "
          f"not a significance test.")
    size_rho, _ = stats.spearmanr(size_perf["final_unique_compounds"],
                                  size_perf["internal_test_roc_auc"])
    print(f"  Dataset size alone: rho = {size_rho:.2f} against test ROC-AUC.")

if globals().get("feature_transferability_df") is not None:
    comb = feature_transferability_df[feature_transferability_df["feature_representation"] == "combined"]
    if len(comb):
        n_all = int((comb["n_targets_in_top_list"] == len(TARGETS)).sum())
        print(f"Question 3 (feature transfer): {n_all} feature(s) reach the top-importance list "
              f"in all {len(TARGETS)} targets.")

if len(ad_error_df):
    n_worse = int((ad_error_df["error_rate_increase_outside_ad"] > 0).sum())
    print(f"Question 4 (uncertainty behavior): error rate rises outside the applicability "
          f"domain for {n_worse}/{len(ad_error_df)} targets.")

if MODULE_G_ENABLED and funnel_df is not None and "n_headline_hits" in funnel_df.columns:
    print(f"Integrated funnel: {int(pd.to_numeric(funnel_df['n_headline_hits'], errors='coerce').sum())} "
          f"prioritized docking hit row(s) across "
          f"{int((pd.to_numeric(funnel_df['n_headline_hits'], errors='coerce') > 0).sum())} target(s).")
else:
    print("Integrated funnel: not assembled (screening/docking outputs unavailable).")

print(f"\nTables and figures: {out_tab_dir} and {out_fig_dir}")
print(f"Meta-analysis artifacts: {meta_path}")
print("Main-text versus supplementary triage is deliberately left to write-up time.")